# Publication Outputs — Manuscript Assembly Notebook

This notebook **assembles every publication artwork and table** — figures, supplementary
figures, tables, supplementary tables and appendices — **in the order in which they appear in
the manuscript**. It is the single place that pins the manuscript's visual and tabular record
together for submission and for public reproduction.

**Design principle — assemble by re-rendering from cached data.**
Nothing here re-runs Google Earth Engine or OSM. Every output is rebuilt from artefacts already
cached under `Results/` by the analysis notebooks, using the plotting functions in
`src/visualization.py`. The analytical logic stays in its home notebooks (`01`–`06`); this
notebook is **assembly only**, so it can be re-run at any time to regenerate the full artwork set.

---

## Notebook structure

| Section | Manuscript section | Status |
|---|---|---|
| **§1** | Methodology | **complete** |
| **§2** | Results — includes the **National** aggregation layer | scaffold (assembled from NB03–NB06) |
| **§3** | Discussion | scaffold |

> The *Introduction* carries no artwork and is omitted.

---

## Output layout — `Results/manuscript_outputs/`
```
Results/manuscript_outputs/
├── main_figures/          Figure_1 … Figure_5
├── supplementary_figures/ Supp_Fig_1.1.* , 1.2.* , 2.* , 3.*
├── tables/                Table_2 , Supp_Table_1
└── appendices/            Appendix_1
```
`Results/` is git-ignored in its entirety, so everything produced here is excluded from version
control and is published separately (e.g. Zenodo) alongside the repository.

---

## Level of analysis — what the supplementary set covers

The study operates at two nested levels — **cluster** and **city**. The supplementary figures
and tables in §1 therefore document those two levels only:

- **Supp Fig 1.1.\*** individual **city** population trajectories;
- **Supp Fig 1.2.\*** individual **cluster** population trends;
- **Supp Fig 2.\*** per-**city** three-zone AOI maps;
- **Supp Fig 3.1 / 3.2** **cluster**- and **city**-level stakeholder maps.

The **National** layer — the aggregation of all cities/clusters — is a *Results* construct and is
assembled in **§2**, not here.

---

## §1 (Methodology) — manuscript item map

| Manuscript item | Content | Source (home notebook) |
|---|---|---|
| **Figure 1** | Study-area overview (Nigeria, AOIs, Cluster25 landscape, proportional population) | NB01 §12a `create_publication_nigeria_overview` |
| **Appendix I** | Cluster25 reference palette — all 24 national PLT classes | national landscape polygon layer + `config.CLUSTER25_COLORS` |
| **Figure 2** | Composite population trend, all AOIs (1975–2050) | NB01 §12c `create_composite_population_trend` |
| **Supp Fig 1.1.1–1.1.67** | Individual **city** population trajectories | WUP city series |
| **Supp Fig 1.2.1–1.2.16** | Individual **cluster** population trends | NB01 `cluster_trends/` |
| **Table 2** | Predominant PLT composition — cities, WUP population, CR & UC areas | NB01 `cluster25_city_summary.csv` + `cluster_zone_area_summary.csv` |
| **Supp Table 1** | Detailed CR + UC cluster distribution per city (%) | NB01 CR/UC distribution |
| **Figure 4** | Five-city AOI delineation grid | NB01 §12b `create_publication_city_grid` |
| **Supp Fig 2.1–2.56** | Per-city three-zone AOI maps | NB01 `{city}_three_zone_map` |
| **Figure 5** | Stakeholder radial (nested-ring) chart | NB02 §6 `create_city_stakeholder_donut` |
| **Supp Fig 3.1 / 3.2** | Cluster- / city-level stakeholder distribution maps | NB02 §5 `create_stakeholder_map_with_bar_chart` |

> **Figure 3** and **Figure 6** are Draw.io schematics — not reproduced here (by design).

**Supplementary-figure numbering** adopted here (now consistent with the manuscript captions):
`1.1.1–1.1.67` (city trends) · `1.2.1–1.2.16` (cluster trends) · `2.1–2.56` (per-city AOI maps) ·
`3.1` (cluster stakeholder map) · `3.2` (city stakeholder map).

# §1 · Methodology

## §1.0 · Environment, paths & output folders

In [ ]:
import sys, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
from shapely.geometry import Point
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

# repo root = parent of this /Notebook folder
ROOT = Path.cwd()
if ROOT.name.lower() == "notebook":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
import config
import visualization as viz
import scs_style as S          # shared: approved cluster labels, colours, checked savefig

warnings.filterwarnings("ignore", message=".*FancyArrowPatch.*")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight",
                     "pdf.fonttype": 42, "ps.fonttype": 42})

# ── input caches ──
RESULTS_AOI = ROOT / "Results" / "01_aoi_delineation"
RESULTS_03  = ROOT / "Results" / "03_temporal_analysis"
RESULTS_04  = ROOT / "Results" / "04_temporal_analysis_daily"
RESULTS_05  = ROOT / "Results" / "05_hotspot_analysis"
RESULTS_06  = ROOT / "Results" / "06_hotspot_modelling"
STUDY_YEAR  = config.PROJECT_STUDY_YEAR

# ── the published v1.0.1 package: the source of record for every value ──
PKG  = RESULTS_05 / "surface_thermal_anomaly_stakeholder_composite"
GPKG = PKG / "surface_thermal_anomaly_stakeholder_composite.gpkg"
assert GPKG.exists(), f"build the package first: {GPKG}"

_LAYERS = {}


def pkg(layer, geom=False):
    """Read a layer of the published package, once."""
    key = (layer, geom)
    if key not in _LAYERS:
        _LAYERS[key] = (gpd.read_file(GPKG, layer=layer) if geom
                        else pyogrio.read_dataframe(GPKG, layer=layer, read_geometry=False))
    return _LAYERS[key].copy()


ASSERTIONS = []


def check(label, ok, detail=""):
    """Record and print an assertion. A figure must not disagree with the package."""
    ASSERTIONS.append((label, bool(ok), detail))
    print(("  [OK]   " if ok else "  [FAIL] ") + label + (f" - {detail}" if detail else ""))
    if not ok:
        raise AssertionError(f"{label}: {detail}")


# ── cluster labels, approved: nothing below spells a cluster name out by hand ──
CL = S.labels()                                   # indexed by integer cluster number
CLUSTER_LABEL      = {int(i): r.label_short for i, r in CL.iterrows()}
CLUSTER_LABEL_LONG = {int(i): r.label_long for i, r in CL.iterrows()}


def cl_header(cid, n=None):
    """Panel header: cluster in words, its code, and how many cities it holds."""
    return S.panel_header(int(cid), n)


# ── output tree ──
OUT       = ROOT / "Results" / "manuscript_outputs"
MAIN_FIG  = OUT / "main_figures"
SUP_FIG   = OUT / "supplementary_figures"
TABLES    = OUT / "tables"
APPENDIX  = OUT / "appendices"
for d in (MAIN_FIG, SUP_FIG, TABLES, APPENDIX):
    d.mkdir(parents=True, exist_ok=True)


def savefig(fig, path_noext, formats=("pdf", "svg"), dpi=300):
    """Save one figure to vector formats and return the paths written.

    format= is passed explicitly and the written magic bytes are checked: relying on the
    extension alone once produced an SVG inside a .pdf, which no visual check would catch.
    """
    out = []
    for ext in formats:
        p = Path(f"{path_noext}.{ext}")
        fig.savefig(p, format=ext, dpi=dpi, bbox_inches="tight")
        head = p.open("rb").read(5)
        want = b"%PDF-" if ext == "pdf" else (b"<?xml" if ext == "svg" else None)
        if want and head != want:
            raise RuntimeError(f"{p.name} was written as {head!r}, not {ext}")
        if p.exists():
            out.append(p)
    return out


print("Repo root :", ROOT)
print("Package   :", GPKG.relative_to(ROOT))
print("Output    :", OUT.relative_to(ROOT))
print("Clusters  :", len(CL), "labelled,", int(CL.has_study_cities.sum()), "with study cities")

## §1.1 · Reconstruct shared inputs from cache

Rebuilds, with **no GEE/OSM calls**:
- `zone_results` — per-city CR / UC / BRA GeoDataFrames (from cached `*_v5.geojson`);
- `cities_in_aoi` — WUP cities falling inside each AOI (spatial join), for the trend charts;
- `pop_override` — authoritative per-AOI population from `zone_area_summary_with_pop.csv`;
- `active_clusters` — the 16 Cluster25 IDs present in the study;
- `landscape_gdf`, `palette` — Cluster25 landscape polygons and reference colours.

In [ ]:
# 1) per-city zone GeoDataFrames -------------------------------------------------
zone_results = {}
for d in sorted(p for p in RESULTS_AOI.iterdir() if p.is_dir()):
    s = d.name
    cr  = d / f"{s}_city_region_4326_v5.geojson"
    uc  = d / f"{s}_urban_core_4326_v5.geojson"
    bra = d / f"{s}_bra_4326_v5.geojson"
    if cr.exists() and uc.exists() and bra.exists():
        zone_results[s.replace("_", " ")] = {
            "city_slug": s, "city_dir": d,
            "cr_gdf": gpd.read_file(cr), "uc_gdf": gpd.read_file(uc), "bra_gdf": gpd.read_file(bra)}
STUDY_CITIES = sorted(zone_results)          # 56 non-absorbed AOIs

# 2) WUP city population table (all candidates) ---------------------------------
wup = pd.read_csv(ROOT / config.AOI_DATA_PATHS["cities_csv"])
YEAR_COLS = [c for c in wup.columns if str(c).isdigit()]
for c in YEAR_COLS:
    wup[c] = wup[c].astype(str).str.replace(",", "", regex=False).astype(float)  # thousands
wup_pts = gpd.GeoDataFrame(
    wup, geometry=[Point(xy) for xy in zip(wup["PWCent_Longitude"], wup["PWCent_Latitude"])],
    crs="EPSG:4326")

# 3) WUP cities inside each AOI (for composite / per-AOI trends) -----------------
cities_in_aoi = {}
for city, res in zone_results.items():
    crg = res["cr_gdf"].to_crs("EPSG:4326")
    inside = gpd.sjoin(wup_pts, crg, predicate="within", how="inner")
    inside = inside.drop(columns=[c for c in inside.columns if c.startswith("index_")], errors="ignore")
    cities_in_aoi[city] = {"cities_in_aoi_df": pd.DataFrame(inside.drop(columns="geometry"))}

# 4) authoritative per-AOI population -------------------------------------------
zpop = pd.read_csv(RESULTS_AOI / "zone_area_summary_with_pop.csv")
pop_override = {str(r["city"]).strip(): float(r["population_wup_k"]) * 1000
               for _, r in zpop.iterrows() if pd.notna(r["population_wup_k"])}

# 5) active Cluster25 IDs --------------------------------------------------------
lookup = pd.read_csv(RESULTS_AOI / "city_cluster25_lookup.csv")
active_clusters = sorted(int(x) for x in lookup["Cluster25"].dropna().unique())

# 6) landscape polygons + reference palette -------------------------------------
landscape_gdf = gpd.read_file(config.LANDSCAPE_DATA_PATH)
palette = pd.read_csv(RESULTS_AOI / "cluster25_reference_palette.csv")

print(f"AOIs (zone_results)      : {len(zone_results)}")
print(f"WUP candidate cities     : {len(wup)}")
print(f"Active Cluster25 IDs (16): {active_clusters}")
print(f"Landscape polygons       : {len(landscape_gdf)}   Palette rows: {len(palette)}")

## §1.2 · Figure 1 — Study-area overview

`create_publication_nigeria_overview` integrates all cartographic elements —
proportional-population circles with a **graduated population legend**, city labels placed close
to their circles, the Cluster25 landscape background, national boundary, **100 km scale bar**,
**north arrow** and cluster legend. Rendered at A4-portrait (8.27 × 11.69 in).

In [ ]:
# ── Fig. 1 · Study area (was v3.0 Figure 1, §1.2) ─────────────────────────────
# Same A4-portrait definition as v3.0: create_publication_nigeria_overview with the
# Cluster25 landscape background, proportional circles, scale bar and north arrow.
# Changed for the SCS rebuild:
#   (a) the 11 absorbed candidates are marked, distinctly from the 56 study cities;
#   (b) the population key starts at the 300,000 selection threshold;
#   (c) the typology legend lists all 24 classes as climate . geology . elevation,
#       with the 16 that hold study cities outlined and marked;
#   (d) no title inside the figure - captions.md carries it;
#   (e) the legend moves beside the map: 24 descriptive labels cannot sit over
#       the country without covering it, as 16 short "Cluster N" labels could.

_prof = pkg("city_profile")
_final = pd.read_csv(RESULTS_AOI / "study_cities_final.csv")

# The absorbed candidates are not in the package (only their names, on the city that
# absorbed them), so their coordinates come from the WUP table. The names are asserted
# back against the package so the two can never drift apart.
_abs = _final[_final.status == "absorbed"].merge(
    wup[["City_Name", "PWCent_Longitude", "PWCent_Latitude"]],
    left_on="city_name", right_on="City_Name", how="left")
_names_pkg = sorted({n.strip() for s in _prof.absorbed_cities.dropna()
                     for n in str(s).split(";")})
check("Fig. 1 - absorbed candidates match the package",
      sorted(_abs.city_name) == _names_pkg,
      f"{len(_abs)} absorbed, {len(_prof)} study cities, {len(_abs) + len(_prof)} candidates")
check("Fig. 1 - every study city is at or above the 300,000 threshold",
      _prof.population_wup_merged_k.min() >= 300,
      f"smallest {_prof.population_wup_merged_k.min():.0f} thousand")

_absorbed_pts = [(r.PWCent_Longitude, r.PWCent_Latitude, r.city_name)
                 for r in _abs.itertuples()]
# Three nested circles, as in v3.0, but keyed to absolute values and starting at the
# 300,000 selection threshold rather than at a fraction of the largest city.
_pop_bands = [(3_000_000, "3 million"), (1_000_000, "1 million"),
              (300_000, "300,000 (threshold)")]
_all_ids = sorted(int(i) for i in CL.index)

fig1 = viz.create_publication_nigeria_overview(
    aoi_dict=zone_results,
    cities_csv_path=ROOT / config.AOI_DATA_PATHS["cities_csv"],
    landscape_path=config.LANDSCAPE_DATA_PATH,
    nigeria_path=config.GHSL_POP_CONFIG["boundary_geojson"],
    cluster25_colors=config.CLUSTER25_COLORS,
    active_cluster_ids=active_clusters,
    study_year=STUDY_YEAR,
    pop_override=pop_override,
    # A4 landscape rather than portrait: Nigeria is roughly square, and the
    # 24-class legend now needs width beside the map rather than height under it.
    save_path=None, dpi=300, figsize=(11.69, 8.27),
    # ── SCS rebuild ──
    cluster_labels={i: f"{CL.loc[i].cluster25}  {CL.loc[i].label_short}"
                    for i in _all_ids},
    legend_cluster_ids=_all_ids,
    highlight_cluster_ids=set(active_clusters),
    absorbed_points=_absorbed_pts,
    pop_legend_values=_pop_bands,
    title=None,
    legend_title="Physiographic landscape typology (PLT)",
    cluster_legend_heading="All 24 classes (climate \u00b7 geology \u00b7 elevation); "
                           "outlined = holds study cities:",
    legend_ncol=1, legend_fontsize=5.8, legend_outside=True)

paths = savefig(fig1, MAIN_FIG / "Fig_01_study_area")
plt.show()
print("Saved:", [p.name for p in paths])

## §1.3 · Appendix I — Physiographic Landscape Typologies (PLT) of Nigeria

Matches the manuscript **Appendix I**: all **24 national PLT classes** (`CL-1 … CL-25`; `CL-2` is
absent nationally) with a **Colour** swatch and their **Climate**, **Geology** and **Elevation**
classes — no land-cover or description columns. Built directly from the national landscape polygon
layer (area-predominant class per cluster); the swatch is the canonical `config.CLUSTER25_COLORS`
value used in Figure 1. *Source: adapted from Eneche et al. (2026).*

In [ ]:
import xlsxwriter

# ── National Cluster25 (PLT) reference — matches manuscript Appendix I ──
# Columns: S/No | Colour | Cluster (CL-X) | Climate Class | Geology Class | Elevation Class.
def _predom(col):
    a = (landscape_gdf.groupby(["Cluster25", col])["area_sqkm"].sum().reset_index()
         .sort_values(["Cluster25", "area_sqkm"], ascending=[True, False]))
    return a.groupby("Cluster25").first()[col]

_nat = pd.DataFrame({
    "Climate Class":   _predom("Climate_Class"),
    "Geology Class":   _predom("Geology_Class"),
    "Elevation Class": _predom("Elevation_Class"),
}).reset_index()
_nat["Cluster25"] = _nat["Cluster25"].astype(int)
_nat = _nat.sort_values("Cluster25").reset_index(drop=True)
_nat["Cluster"] = _nat["Cluster25"].map(lambda c: f"CL-{c}")
_nat["Colour"]  = _nat["Cluster25"].map(lambda c: config.CLUSTER25_COLORS.get(int(c), "#CCCCCC"))
_nat.insert(0, "S/No", range(1, len(_nat) + 1))

# CSV carries the hex under "Colour"; the inline preview and Excel render it as a swatch.
appx = _nat[["S/No", "Cluster", "Climate Class", "Geology Class", "Elevation Class", "Colour"]]

_styler = (_nat[["S/No", "Colour", "Cluster", "Climate Class", "Geology Class", "Elevation Class"]]
           .style
           .set_caption("Appendix I — Physiographic Landscape Typologies in Nigeria and their internal makeup")
           .set_table_styles([
               {"selector": "th", "props": [("background-color", "#404040"),
                                            ("color", "white"), ("font-weight", "bold")]},
               {"selector": "caption", "props": [("font-weight", "bold"),
                                                 ("font-size", "13px"), ("text-align", "left")]},
           ])
           .set_properties(**{"font-size": "11px"})
           .apply(lambda s: [f"background-color:{v}; color:{v}; border:1px solid #999" for v in s],
                  subset=["Colour"], axis=0)
           .hide(axis="index"))
display(_styler)

# ── (a) CSV (Colour column carries the hex) ──
csv_p = APPENDIX / "Appendix_1_PLT_reference.csv"
appx.to_csv(csv_p, index=False)

# ── (b) Excel — Colour rendered as a filled swatch (no text) ──
xlsx_p = APPENDIX / "Appendix_1_PLT_reference.xlsx"
wb = xlsxwriter.Workbook(str(xlsx_p))
ws = wb.add_worksheet("Appendix I")
hdr  = wb.add_format({"bold": True, "bg_color": "#404040", "font_color": "white",
                      "border": 1, "align": "center", "valign": "vcenter", "text_wrap": True})
cell = wb.add_format({"border": 1, "valign": "vcenter", "text_wrap": True})
cols = ["S/No", "Colour", "Cluster", "Climate Class", "Geology Class", "Elevation Class"]
for j, c in enumerate(cols):
    ws.write(0, j, c, hdr)
for i, (_, r) in enumerate(_nat.iterrows(), start=1):
    hx = str(r["Colour"])
    ws.write(i, 0, int(r["S/No"]), cell)
    ws.write(i, 1, "", wb.add_format({"bg_color": hx, "border": 1}))
    ws.write(i, 2, r["Cluster"], cell)
    ws.write(i, 3, r["Climate Class"], cell)
    ws.write(i, 4, r["Geology Class"], cell)
    ws.write(i, 5, r["Elevation Class"], cell)
ws.set_column(0, 0, 6); ws.set_column(1, 1, 9); ws.set_column(2, 2, 9); ws.set_column(3, 5, 30)
ws.set_row(0, 26)
wb.close()
print(f"Saved {len(appx)}-cluster Appendix I (manuscript format):", csv_p.name, "+", xlsx_p.name)

## §1.4 · Figure 2 — Composite population trend

Composite WUP trajectory (1975–2050) across all AOIs: total trend with 2024/2050 markers, linear
fit, per-city top lines and a decadal growth-rate inset. Rendered landscape (11.69 × 5.845 in) for
a full-width manuscript figure.

In [ ]:
fig2, stats2 = viz.create_composite_population_trend(
    cities_in_aoi, save_dir=None, figsize=(11.69, 5.845), dpi=300,
    mode="aoi", all_cities_df=wup)

paths = savefig(fig2, MAIN_FIG / "Figure_2_population_trend")
plt.show()
print("Saved:", [p.name for p in paths])
print("Totals — 2024: {pop_2024_k:,.0f}k over {n_aois} AOIs".format(**stats2))

## §1.5 · Supplementary Figures 1.1.1 – 1.1.67 — individual city trajectories

One panel per **WUP candidate city** (all 67, no roll-up), numbered `1.1.1 … 1.1.67` in the order
they appear in the WUP table. Uniform house style: navy trajectory, crimson 2024 marker, green 2050
marker. Written to `supplementary_figures/1.1_city_trends/`.

In [ ]:
CITY_TREND_DIR = SUP_FIG / "1.1_city_trends"
CITY_TREND_DIR.mkdir(exist_ok=True)

years5 = [int(y) for y in YEAR_COLS if int(y) % 5 == 0]
def plot_city_trend(row, save_noext):
    ys = [row[str(y)] for y in years5]            # thousands
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    ax.plot(years5, ys, color="navy", marker="s", ms=3.5, lw=1.2, alpha=0.8)
    for yr, col in ((2024, "crimson"), (2050, "darkgreen")):
        if str(yr) in row.index:
            ax.axvline(yr, color=col, ls="--", lw=0.9)
            ax.scatter(yr, row[str(yr)], marker="D", s=55, color=col,
                       edgecolors="black", lw=0.8, zorder=5)
    ax.set_title(str(row["City_Name"]), fontweight="bold", fontsize=11)
    ax.set_xlabel("Year"); ax.set_ylabel("Population (thousands)")
    ax.grid(alpha=0.25); fig.tight_layout()
    p = savefig(fig, save_noext, formats=("pdf", "svg"))
    plt.close(fig); return p

trend_map = []
for i, (_, row) in enumerate(wup.iterrows(), start=1):
    tag = f"1.1.{i}"
    slug = str(row["City_Name"]).replace(" ", "_").replace("/", "-")
    plot_city_trend(row, CITY_TREND_DIR / f"Supp_Fig_{tag}_{slug}")
    trend_map.append({"supp_fig": tag, "city": row["City_Name"]})
pd.DataFrame(trend_map).to_csv(CITY_TREND_DIR / "_index_1.1_city_trends.csv", index=False)
print(f"Generated {len(trend_map)} city-trend figures  ->  {CITY_TREND_DIR.relative_to(OUT)}")

## §1.6 · Supplementary Figures 1.2.1 – 1.2.16 — cluster population trends

Per-cluster trends already cached as vector PDFs/SVGs under `Results/01_aoi_delineation/cluster_trends/`.
Collected and renumbered `1.2.1 … 1.2.16` in ascending Cluster25 order; a mapping CSV records the
`Supp Fig ↔ Cluster25` correspondence.

In [ ]:
CLUS_TREND_DIR = SUP_FIG / "1.2_cluster_trends"
CLUS_TREND_DIR.mkdir(exist_ok=True)
src_dir = RESULTS_AOI / "cluster_trends"

clus_map = []
for i, cid in enumerate(active_clusters, start=1):
    tag = f"1.2.{i}"
    picked = 0
    for ext in ("pdf", "svg"):
        src = src_dir / f"cluster_{cid:02d}_population_trend.{ext}"
        if src.exists():
            shutil.copyfile(src, CLUS_TREND_DIR / f"Supp_Fig_{tag}_C{cid}.{ext}"); picked += 1
    clus_map.append({"supp_fig": tag, "Cluster25": f"C{cid}", "found_files": picked})
pd.DataFrame(clus_map).to_csv(CLUS_TREND_DIR / "_index_1.2_cluster_trends.csv", index=False)
print(f"Collected {len(clus_map)} cluster-trend figures  ->  {CLUS_TREND_DIR.relative_to(OUT)}")
display(pd.DataFrame(clus_map))

## §1.7 · Table 2 — Predominant PLT composition of the city-regions

Cluster-level composition matching the manuscript table: member **cities**, **frequency** and
**share**, **WUP population**, and **City-Region (CR)** and **Urban-Core (UC)** areas, with the
physiographic description. Built by joining `cluster25_city_summary.csv` (city membership) with
`cluster_zone_area_summary.csv` (population + areas). Exported as CSV and colour-keyed Excel.

In [ ]:
_summary = pd.read_csv(RESULTS_AOI / "cluster25_city_summary.csv")
_areas   = pd.read_csv(RESULTS_AOI / "cluster_zone_area_summary.csv")
hexmap   = dict(zip(palette["Cluster25"].astype(str), palette["Hex Colour"].astype(str)))

_summary["_cid"] = _summary["Physiographic Cluster"].str.lstrip("C").astype(int)
_areas = _areas.rename(columns={"Cluster25": "_cid"})
tab2 = _summary.merge(
    _areas[["_cid", "total_population_wup_k", "total_cr_area_km2",
            "total_uc_area_km2", "total_bra_area_km2"]],
    on="_cid", how="left")

tab2 = tab2.rename(columns={
    "S/No.": "S/No", "City Regions": "City Regions",
    "Physiographic Cluster": "Cluster", "Frequency": "N Cities",
    "Percentage (%)": "% of Cities",
    "total_population_wup_k": "WUP Population (thousands)",
    "total_cr_area_km2": "CR Area (km2)",
    "total_uc_area_km2": "UC Area (km2)",
})
tab2 = tab2[["S/No", "Cluster", "City Regions", "N Cities", "% of Cities",
             "WUP Population (thousands)", "CR Area (km2)", "UC Area (km2)", "Description"]]
for c in ["WUP Population (thousands)", "CR Area (km2)", "UC Area (km2)"]:
    tab2[c] = tab2[c].round(1)

csv2 = TABLES / "Table_2_plt_composition.csv"
tab2.to_csv(csv2, index=False)

xls2 = TABLES / "Table_2_plt_composition.xlsx"
wb = xlsxwriter.Workbook(str(xls2)); ws = wb.add_worksheet("Table 2")
hdr  = wb.add_format({"bold": True, "bg_color": "#404040", "font_color": "white",
                      "border": 1, "align": "center", "valign": "vcenter", "text_wrap": True})
cell = wb.add_format({"border": 1, "valign": "vcenter", "text_wrap": True})
num  = wb.add_format({"border": 1, "valign": "vcenter", "num_format": "#,##0.0"})
outcols = list(tab2.columns) + ["Colour"]
for j, c in enumerate(outcols):
    ws.write(0, j, c, hdr)
_numcols = {"WUP Population (thousands)", "CR Area (km2)", "UC Area (km2)"}
for i, (_, r) in enumerate(tab2.iterrows(), start=1):
    for j, c in enumerate(tab2.columns):
        ws.write(i, j, r[c], num if c in _numcols else cell)
    ws.write(i, len(tab2.columns), "",
             wb.add_format({"bg_color": hexmap.get(str(r["Cluster"]), "#FFFFFF"), "border": 1}))
ws.set_column(0, 0, 6); ws.set_column(1, 1, 9); ws.set_column(2, 2, 58)
ws.set_column(3, 4, 10); ws.set_column(5, 7, 16); ws.set_column(8, 8, 52); ws.set_column(9, 9, 9)
wb.close()
print("Saved:", csv2.name, "+", xls2.name)
display(tab2)

## §1.8 · Supplementary Table 1 — Detailed CR + UC cluster distribution

Per-city percentage composition of every Cluster25 within the **City-Region (CR)** and
**Urban-Core (UC)** zones (NB01 "Detailed Distribution & Colour Reference"). Cities as rows. A
combined long-form CSV plus a two-sheet (CR, UC) Excel workbook with colour-keyed headers.

In [ ]:
cr = pd.read_csv(RESULTS_AOI / "cluster25_cr_distribution.csv")
uc = pd.read_csv(RESULTS_AOI / "cluster25_uc_distribution.csv")

def _melt(df, zone):
    m = df.melt(id_vars="City Region", var_name="Cluster25", value_name="pct")
    m.insert(1, "Zone", zone)
    return m[m["pct"] > 0]
combined = pd.concat([_melt(cr, "CR"), _melt(uc, "UC")], ignore_index=True)
csvS1 = TABLES / "Supp_Table_1_detailed_distribution.csv"
combined.to_csv(csvS1, index=False)

xlsS1 = TABLES / "Supp_Table_1_detailed_distribution.xlsx"
wb = xlsxwriter.Workbook(str(xlsS1))
hdrbase = {"bold": True, "border": 1, "align": "center", "valign": "vcenter", "font_size": 9}
cellf = wb.add_format({"border": 1, "num_format": "0.0", "font_size": 9})
namef = wb.add_format({"border": 1, "bold": True, "font_size": 9})
for zone, df in (("CR", cr), ("UC", uc)):
    ws = wb.add_worksheet(zone)
    ws.write(0, 0, "City Region",
             wb.add_format({**hdrbase, "bg_color": "#404040", "font_color": "white"}))
    for j, cl in enumerate(df.columns[1:], start=1):
        ws.write(0, j, cl, wb.add_format({**hdrbase, "bg_color": hexmap.get(str(cl), "#DDDDDD")}))
    for i, (_, r) in enumerate(df.iterrows(), start=1):
        ws.write(i, 0, r["City Region"], namef)
        for j, cl in enumerate(df.columns[1:], start=1):
            v = r[cl]
            ws.write(i, j, v if v > 0 else None, cellf)
    ws.set_column(0, 0, 16); ws.set_column(1, len(df.columns), 5); ws.freeze_panes(1, 1)
wb.close()
print("Saved:", csvS1.name, "+", xlsS1.name, "| combined rows:", len(combined))
display(combined.head(10))

## §1.9 · Figure 4 — Five-city AOI delineation grid

Representative panel of Kano, Abuja, Jos, Lagos and Port Harcourt — each showing CR / BRA / UC
zones over the Cluster25 landscape, with water and roads. A **single north arrow** is placed just
above the shared legend (`single_north_arrow=True`) rather than one per panel.

In [ ]:
# ── Fig. 3 · Delineation excerpt (was v3.0 Figure 4, §1.9) ─────
# Unchanged definition; only the output name follows the new numbering.
GRID_CITIES = ["Kano", "Abuja", "Jos", "Lagos", "Port Harcourt"]
fig4 = viz.create_publication_city_grid(
    city_names=GRID_CITIES, zone_results=zone_results, results_aoi_path=RESULTS_AOI,
    landscape_gdf=landscape_gdf, cluster25_colors=config.CLUSTER25_COLORS,
    storage_crs="EPSG:4326", save_path=None, dpi=300, figsize=(8.27, 5.845),
    single_north_arrow=True)

paths = savefig(fig4, MAIN_FIG / "Fig_03_delineation_excerpt")
plt.show()
print("Saved:", [p.name for p in paths])

## §1.10 · Supplementary Figures 2.1 – 2.56 — per-city AOI maps

The individual three-zone (CR / BRA / UC) delineation map for every study city, cached as vector
PDFs/SVGs (`{city}_three_zone_map`). Collected and renumbered `2.1 … 2.56` (alphabetical), with a
`Supp Fig ↔ city` index. These are the per-city children of Figure 4.

In [ ]:
AOI_MAP_DIR = SUP_FIG / "2_city_aoi_maps"
AOI_MAP_DIR.mkdir(exist_ok=True)

aoi_map = []
for i, city in enumerate(STUDY_CITIES, start=1):
    tag = f"2.{i}"
    slug = zone_results[city]["city_slug"]
    fig_dir = zone_results[city]["city_dir"] / "figures"
    found = 0
    for ext in ("pdf", "svg"):
        src = fig_dir / f"{slug}_three_zone_map.{ext}"
        if src.exists():
            shutil.copyfile(src, AOI_MAP_DIR / f"Supp_Fig_{tag}_{slug}.{ext}"); found += 1
    aoi_map.append({"supp_fig": tag, "city": city, "found_files": found})
pd.DataFrame(aoi_map).to_csv(AOI_MAP_DIR / "_index_2_city_aoi_maps.csv", index=False)
print(f"Collected {len(aoi_map)} per-city AOI maps  ->  {AOI_MAP_DIR.relative_to(OUT)}")
display(pd.DataFrame(aoi_map).head())

## §1.11 · Figure 5 & Supplementary Figures 3.1 / 3.2 — stakeholder items

These originate in `02_stakeholder_analysis.ipynb`.

- **Figure 5** is **re-rendered** here from the cached NB02 stakeholder tables via
  `create_city_stakeholder_donut` — the nested-ring radial (Ring 1 clusters → Ring 2 attendance →
  Ring 3 cities). Center reads *"N Target Cities / M Workshop Participants"*; the legend lists each
  cluster with a **filled** marker (has attendance) or **open** marker (none) as `CX (N cities | M att.)`.
- **Supp Fig 3.1 / 3.2** — cluster- and city-level stakeholder distribution maps — are collected
  as-is from the cached NB02 renders.

In [ ]:
STAKE     = ROOT / "Results" / "02_stakeholder_analysis"
STAKE_FIG = STAKE / "Figures"
STAKE_OUT = SUP_FIG / "3_stakeholder_maps"
STAKE_OUT.mkdir(exist_ok=True)

# ── Figure 5 — re-render from cached stakeholder tables (no survey re-processing) ──
cluster_stake_df = pd.read_csv(STAKE / "cluster25_stakeholder_distribution.csv")
_stk_summary = pd.read_csv(STAKE / "Data" / "Cities_Stakeholder_Summary_2024.csv")
_absorbed = set(pd.read_csv(RESULTS_AOI / "absorbed_cities.csv")["absorbed_city"])
study_stake_gdf = _stk_summary[~_stk_summary["city"].isin(_absorbed)].copy()   # 56 study cities
study_stake_gdf["Cluster25"] = study_stake_gdf["Cluster25"].astype(int)
_attended_pids = pd.read_csv(STAKE / "Data" / "workshop_attendees_clean.csv")["pid"].nunique()

fig5 = viz.create_city_stakeholder_donut(
    cluster_summary=cluster_stake_df, study_gdf=study_stake_gdf,
    cluster25_colors=config.CLUSTER25_COLORS, attended_pids=_attended_pids, figsize=(16, 9))
paths5 = savefig(fig5, MAIN_FIG / "Figure_5_stakeholder_radial")
plt.show()
print(f"Figure 5: {len(study_stake_gdf)} target cities | {_attended_pids} workshop participants "
      f"-> {[p.name for p in paths5]}")

# ── Supp Fig 3.1 / 3.2 — collected as-is from cached NB02 renders ──
_nb02_maps = [
    ("Supp Fig 3.1", "cluster25_stakeholder_map",         "Supp_Fig_3.1_cluster_stakeholder_map"),
    ("Supp Fig 3.2", "Stakeholder_Distribution_Map_2024", "Supp_Fig_3.2_city_stakeholder_map"),
]
for label, srcbase, dstbase in _nb02_maps:
    picked = []
    for ext in ("pdf", "svg", "png"):
        src = STAKE_FIG / f"{srcbase}.{ext}"
        if src.exists():
            shutil.copyfile(src, STAKE_OUT / f"{dstbase}.{ext}"); picked.append(ext)
    print(f"{label:14} <- {srcbase:40} [{', '.join(picked) or 'MISSING'}]")

## §1.12 · Methodology outputs — summary

In [ ]:
print("Methodology artwork written under", OUT.relative_to(ROOT))
for sub in ["main_figures", "supplementary_figures", "tables", "appendices"]:
    d = OUT / sub
    n = sum(1 for p in d.rglob("*") if p.is_file() and not p.name.startswith("_index"))
    print(f"  {sub:24}: {n} files")

# §2 · Results

Figure and table numbers below follow the **manuscript** (this notebook is the source of truth for
numbering). §2.1 covers the long-term (2006–2025) trend analysis (NB03): **Figure 7** (annual) and
**Figure 8** (monthly) at cluster level for the selected clusters, with the complete cluster / city /
national levels in **Supplementary Figures 4.1–4.3** (annual) and **5.1–5.3** (monthly).

## §2.1 · Long-term trend — Figures 7 & 8, Supp 4 & 5

Re-rendered from the cached NB03 series. The combined per-entity chart is **split** into an
annual-only panel (Figure 7 / Supp 4) and a monthly-only panel (Figure 8 / Supp 5) via
`src/temporal_panels.py`. Each panel keeps day+night UC/BRA, SUHII (right axis) with Sen-slope
overlay, CR P5–P95 band, peak/trough markers and MK/Sen stat boxes. Colours: **day** = orange/gold/red,
**night** = black/grey/purple. Selected clusters for the main figures: **C4, C7, C9, C13, C19, C25**.

In [ ]:
import math
import matplotlib.patches as _mpat
from matplotlib.lines import Line2D as _L2D
import temporal_panels as tp

RESULTS_03 = ROOT / "Results" / "03_temporal_analysis"
STATS03    = RESULTS_03 / "Cluster25_Composites" / "stats"

SELECTED_CLUSTERS = [4, 7, 9, 13, 19, 25]
_cz = pd.read_csv(RESULTS_AOI / "cluster_zone_area_summary.csv")
CL_META = {int(r["Cluster25"]): (r["climate_class"], int(r["n_cities"])) for _, r in _cz.iterrows()}

def _temporal_legend(fig, y):
    Cc = tp.NB03_COLOURS
    fig.legend(handles=[
        _L2D([0],[0], color=Cc['uc_day'],    lw=1.4, label='UC — day'),
        _L2D([0],[0], color=Cc['uc_night'],  lw=1.4, label='UC — night'),
        _L2D([0],[0], color=Cc['bra_day'],   lw=1.4, label='BRA — day'),
        _L2D([0],[0], color=Cc['bra_night'], lw=1.4, label='BRA — night'),
        _L2D([0],[0], color=Cc['delta_day'], lw=1.8, label='SUHII — day'),
        _L2D([0],[0], color=Cc['delta_night'],lw=1.8, label='SUHII — night'),
        _mpat.Patch(facecolor=Cc['cr_day_fill'],   alpha=.55, label='CR range — day'),
        _mpat.Patch(facecolor=Cc['cr_night_fill'], alpha=.55, label='CR range — night'),
        _L2D([0],[0], color='#777', lw=.9, ls='--', label="Sen's slope"),
    ], loc='center', ncol=5, fontsize=7, framealpha=.9, bbox_to_anchor=(0.5, y))

def _cluster_title(cid):
    clim, n = CL_META.get(cid, ("", 0)); return f"C{cid} — {clim} ({n} cities)"

def _trend_cell(stats, dstats, zone, period):
    src = dstats if zone == "delta_uc_bra" else stats
    if src is None or src.empty:
        return "—"
    r = src[(src["zone"] == zone) & (src["period"] == period)]
    if r.empty:
        return "—"
    r = r.iloc[0]
    sym = {"increasing": "↑", "decreasing": "↓"}.get(str(r.get("mk_trend", "")), "—")
    try:
        pf = float(r.get("mk_p")); sig = "***" if pf < .001 else ("**" if pf < .01 else ("*" if pf < .05 else "ns"))
    except (TypeError, ValueError):
        sig = ""
    try:
        dec = f"{float(r.get('sen_slope_per_decade')):+.2f}"
    except (TypeError, ValueError):
        dec = ""
    hr = "ᴴ" if r.get("autocorr_corrected", False) else ""
    return f"{sym}{sig} {dec}{hr}".strip()

def render_cluster_grid(cids, kind, save_noext):
    """2-col × 3-row cluster grid (no in-panel stat boxes) + a summary table below the legend."""
    fig = plt.figure(figsize=(8.27, 11.69))
    gs = fig.add_gridspec(3, 2, top=0.965, bottom=0.345, left=0.10, right=0.90,
                          hspace=0.55, wspace=0.50)
    trows = []
    for k, cid in enumerate(cids):
        ax = fig.add_subplot(gs[k // 2, k % 2]); ax2 = ax.twinx()
        df, ddf, st, ds = tp.panel_inputs(tp.load_cluster(cid, STATS03), kind)
        tp.draw_temporal_panel(ax, ax2, kind, df, ddf, st, ds, stat_boxes=False)
        ax.set_title(_cluster_title(cid), fontsize=8, fontweight="bold", pad=3)
        trows.append([f"C{cid}",
                      _trend_cell(st, ds, "urban_core", "day"),  _trend_cell(st, ds, "bra", "day"),  _trend_cell(st, ds, "delta_uc_bra", "day"),
                      _trend_cell(st, ds, "urban_core", "night"),_trend_cell(st, ds, "bra", "night"),_trend_cell(st, ds, "delta_uc_bra", "night")])
    _temporal_legend(fig, y=0.30)
    # ── in-panel summary table (holds the per-cluster trend detail; not referenced elsewhere) ──
    tax = fig.add_axes([0.05, 0.035, 0.90, 0.215]); tax.axis("off")
    cols = ["Cluster", "UC · Day", "BRA · Day", "SUHII · Day", "UC · Night", "BRA · Night", "SUHII · Night"]
    tbl = tax.table(cellText=trows, colLabels=cols, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(6.4); tbl.scale(1, 1.55)
    for (rr, cc), cl in tbl.get_celld().items():
        cl.set_edgecolor("#cccccc")
        if rr == 0:
            cl.set_facecolor("#404040"); cl.set_text_props(color="white", fontweight="bold")
        elif cc == 0:
            cl.set_text_props(fontweight="bold")
    tax.set_title("Trend summary — ↑/↓ Mann-Kendall direction · */**/*** = p<.05/.01/.001 · "
                  "value = Sen slope (°C/decade) · ᴴ = Hamed–Rao corrected",
                  fontsize=6.5, pad=4)
    return savefig(fig, save_noext)



# ── SCS rebuild ───────────────────────────────────────────────────────────────
# Trend values are taken from the published composite_trend_statistics layer rather
# than from the NB03 stats CSVs, and the two are asserted to agree first. Zone names
# differ between the two: tp uses urban_core / bra / delta_uc_bra, the package uses
# uc / bra / suhii.
_ZONE_PKG = {"urban_core": "uc", "bra": "bra", "delta_uc_bra": "suhii"}
_COMP = pkg("composite_trend_statistics")
_COMP = _COMP[_COMP.composite_type == "cluster"]


def _pkg_trend(cid, zone, period):
    """One row of the published cluster composite, by tp zone name."""
    r = _COMP[(_COMP.composite_key == f"C{int(cid)}")
              & (_COMP.zone == _ZONE_PKG[zone]) & (_COMP.period == period)]
    return None if r.empty else r.iloc[0]


def _trend_cell_pkg(cid, zone, period):
    """Same cell format as _trend_cell, but sourced from the package."""
    r = _pkg_trend(cid, zone, period)
    if r is None:
        return "\u2014"
    sym = {"increasing": "\u2191", "decreasing": "\u2193"}.get(str(r.mk_trend), "\u2014")
    p = float(r.mk_p_value)
    sig = "***" if p < .001 else ("**" if p < .01 else ("*" if p < .05 else "ns"))
    hr = "\u1d34" if bool(r.autocorr_corrected) else ""
    return f"{sym}{sig} {float(r.sen_slope_c_per_decade):+.2f}{hr}".strip()


def assert_trends_match_package(cids, tol=1e-5):
    """The NB03 stats behind the panels and the published composites must agree."""
    worst, n = 0.0, 0
    for cid in cids:
        ent = tp.load_cluster(cid, STATS03)
        _, _, st, ds = tp.panel_inputs(ent, "annual")
        for zone in ("urban_core", "bra", "delta_uc_bra"):
            src = ds if zone == "delta_uc_bra" else st
            if src is None or src.empty:
                continue
            for period in ("day", "night"):
                row = src[(src["zone"] == zone) & (src["period"] == period)]
                pk = _pkg_trend(cid, zone, period)
                if row.empty or pk is None:
                    continue
                a = float(row.iloc[0]["sen_slope_per_decade"])
                b = float(pk.sen_slope_c_per_decade)
                worst = max(worst, abs(a - b)); n += 1
    check("Fig. 4 - NB03 panel statistics match composite_trend_statistics",
          worst <= tol, f"{n} slopes compared, worst difference {worst:.2e}")


def render_fig04(cids, save_noext):
    """Fig. 4: v3.0 Figures 7 and 8 merged onto one A4 page.

    Same panel definition as v3.0 (tp.draw_temporal_panel, with its secondary SUHII
    axis), same legend and same trend-summary table. Changed: annual and monthly sit
    side by side for each cluster, headers use the approved cluster labels, and the
    table is filled from the published composites.
    """
    assert_trends_match_package(cids)
    fig = plt.figure(figsize=(8.27, 11.69))
    gs = fig.add_gridspec(len(cids), 2, top=0.975, bottom=0.300, left=0.095, right=0.905,
                          hspace=0.62, wspace=0.52)
    trows = []
    for k, cid in enumerate(cids):
        ent = tp.load_cluster(cid, STATS03)
        for j, kind in enumerate(("annual", "monthly")):
            ax = fig.add_subplot(gs[k, j]); ax2 = ax.twinx()
            df, ddf, st, ds = tp.panel_inputs(ent, kind)
            tp.draw_temporal_panel(ax, ax2, kind, df, ddf, st, ds, stat_boxes=False)
            if j == 0:
                ax.set_title(cl_header(cid), fontsize=7.6, fontweight="bold", pad=3,
                             loc="left")
            else:
                ax.set_title("Monthly climatology", fontsize=7.6, fontweight="bold",
                             pad=3, loc="left")
        trows.append([f"C{cid}",
                      _trend_cell_pkg(cid, "urban_core", "day"),
                      _trend_cell_pkg(cid, "bra", "day"),
                      _trend_cell_pkg(cid, "delta_uc_bra", "day"),
                      _trend_cell_pkg(cid, "urban_core", "night"),
                      _trend_cell_pkg(cid, "bra", "night"),
                      _trend_cell_pkg(cid, "delta_uc_bra", "night")])
    _temporal_legend(fig, y=0.258)
    tax = fig.add_axes([0.05, 0.030, 0.90, 0.195]); tax.axis("off")
    cols = ["Cluster", "UC \u00b7 Day", "BRA \u00b7 Day", "SUHII \u00b7 Day",
            "UC \u00b7 Night", "BRA \u00b7 Night", "SUHII \u00b7 Night"]
    tbl = tax.table(cellText=trows, colLabels=cols, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(6.4); tbl.scale(1, 1.5)
    for (rr, cc), cl in tbl.get_celld().items():
        cl.set_edgecolor("#cccccc")
        if rr == 0:
            cl.set_facecolor("#404040"); cl.set_text_props(color="white", fontweight="bold")
        elif cc == 0:
            cl.set_text_props(fontweight="bold")
    tax.set_title("Trend summary from the published cluster composites \u2014 "
                  "\u2191/\u2193 Mann-Kendall direction \u00b7 */**/*** = p<.05/.01/.001 "
                  "\u00b7 value = Sen slope (\u00b0C/decade) \u00b7 "
                  "\u1d34 = Hamed\u2013Rao corrected", fontsize=6.3, pad=4)
    return savefig(fig, save_noext)


print("Temporal setup ready — clusters:", len(active_clusters), "| selected (Fig 7/8):", SELECTED_CLUSTERS)

### Fig. 4 — Long-term trends: annual series and monthly climatology

v3.0 Figures 7 and 8 merged onto one A4 page, one row per selected cluster.

In [ ]:
# ── Fig. 4 · Long-term trends (was v3.0 Figures 7 + 8, §2.1 cells 32 and 34) ──
paths = render_fig04(SELECTED_CLUSTERS, MAIN_FIG / "Fig_04_longterm_trends")
plt.show(); print("Fig. 4:", [p.name for p in paths])

### (v3.0 Figure 8 — now the right-hand column of Fig. 4)

In [ ]:
# v3.0 Figure 8 (monthly climatology) is now the right-hand column of Fig. 4, rendered
# in the cell above. render_cluster_grid() is kept for the supplementary trend figures.
print("Monthly climatology is the right-hand column of Fig. 4 (see the cell above).")

### Supplementary — complete cluster / city / national trends

The **full** NB03 per-entity temporal charts (annual **and** monthly together) are collected here
**as-is**, with no re-computation. Figures 7 and 8 above are the manuscript panels (annual-only and
monthly-only for the selected clusters); these supplementary charts are the original combined form.

- **Supp Fig 4.1.1 – 4.1.16** — per **cluster** (all 16)
- **Supp Fig 4.2.1 – 4.2.56** — per **city** (all 56)
- **Supp Fig 4.3** — **national** composite

> Each combined chart holds annual (left) and monthly (right), so the manuscript's separate
> Figure 7 (Supp 4.x) and Figure 8 (Supp 5.x) references both map to this single **4.x** set — see
> the mapping note printed below.

In [ ]:
SUP_T = SUP_FIG / "4_temporal_trends"; SUP_T.mkdir(exist_ok=True)
CL_T  = SUP_T / "4.1_cluster"; CL_T.mkdir(exist_ok=True)
CT_T  = SUP_T / "4.2_city";    CT_T.mkdir(exist_ok=True)

# ── Supp 4.1 — 16 cluster combined charts (collected) ──
_ci = []
for i, cid in enumerate(active_clusters, start=1):
    sd = RESULTS_03 / "Cluster25_Composites" / f"Cluster25_{cid:02d}" / "figures"
    for ext in ("pdf", "svg"):
        s = sd / f"Cluster25_{cid:02d}_temporal_trend.{ext}"
        if s.exists():
            shutil.copyfile(s, CL_T / f"Supp_Fig_4.1.{i}_C{cid}.{ext}")
    _ci.append({"supp_fig": f"4.1.{i}", "Cluster25": f"C{cid}"})
pd.DataFrame(_ci).to_csv(CL_T / "_index_4.1_cluster.csv", index=False)

# ── Supp 4.2 — 56 city combined charts (collected) ──
_cy = []
for i, city in enumerate(STUDY_CITIES, start=1):
    slug = zone_results[city]["city_slug"]; sd = RESULTS_03 / slug / "figures"
    found = 0
    for ext in ("pdf", "svg"):
        s = sd / f"{slug}_temporal_trend.{ext}"
        if s.exists():
            shutil.copyfile(s, CT_T / f"Supp_Fig_4.2.{i}_{slug}.{ext}"); found += 1
    _cy.append({"supp_fig": f"4.2.{i}", "city": city, "found_files": found})
pd.DataFrame(_cy).to_csv(CT_T / "_index_4.2_city.csv", index=False)

# ── Supp 4.3 — national composite (collected) ──
_ncf = RESULTS_03 / "National_Composite" / "figures"
for ext in ("pdf", "svg"):
    s = _ncf / f"national_composite_temporal_trend.{ext}"
    if s.exists():
        shutil.copyfile(s, SUP_T / f"Supp_Fig_4.3_national.{ext}")

print(f"Collected combined trends -> Supp 4.1 (16 clusters), 4.2 ({len(_cy)} cities), 4.3 (national)")
print("MAIN-TEXT MAPPING: Fig 7 & Fig 8 both point to Supp 4.1 (cluster) / 4.2 (city) / 4.3 (national).")

## §2.2 · Stakeholder perception & corroboration — Figure 9, Supp Fig 6, Table 4, Supp Table 3

- **Figure 9** — stakeholder perception at **cluster** level (warming / diurnal severity / peak
  season); the centre donut reports the national total, so no separate national figure is needed.
- **Supp Fig 6** — the same at **city** level.
- **Table 4** — stakeholder corroboration vs satellite UC trend, for the **21 represented cities**
  (+ their clusters + national aggregate).
- **Supp Table 3** — the **complete** version: all 56 cities and 16 clusters, including those
  without stakeholder representation.

> **Note — participant base.** The perception items cover the **105**-participant cohort, which is
> defined by the *registered* city; one respondent who attended without registering is excluded as
> unverifiable. Figure 18 (national hit rate) is the NB05 output, which keys on attendance and so
> rests on 97 GPS points rather than the cohort's 94 — see the note in NB05 §2c.


In [ ]:
# ── Figure 9 (cluster perception) + Supp Fig 6 (city perception) — collected from NB03 ──
PERC = RESULTS_03 / "Perception_Panels"
SUP6 = SUP_FIG / "6_city_perception"; SUP6.mkdir(exist_ok=True)
for label, srcbase, dst, dstbase in [
    ("Figure 9",  "Perception_Cluster_3panel", MAIN_FIG, "Figure_9_stakeholder_perception"),
    ("Supp Fig 6", "Perception_City_3panel",   SUP6,     "Supp_Fig_6_city_perception"),
]:
    picked = []
    for ext in ("pdf", "svg", "png"):
        s = PERC / f"{srcbase}.{ext}"
        if s.exists():
            shutil.copyfile(s, dst / f"{dstbase}.{ext}"); picked.append(ext)
    print(f"{label:11} <- {srcbase:30} [{', '.join(picked) or 'MISSING'}]")

In [ ]:
# ── Table 4 — stakeholder corroboration (represented cities + clusters + national) ──
comp = pd.read_csv(RESULTS_03 / "Stakeholder_Validation_Comprehensive.csv")
_order = pd.Categorical(comp["Level"], categories=["City", "Cluster", "National"], ordered=True)
tab4 = comp.assign(_o=_order).sort_values(["_o", "Cluster25", "Unit"]).drop(columns="_o").reset_index(drop=True)
csv4 = TABLES / "Table_4_stakeholder_corroboration.csv"
tab4.to_csv(csv4, index=False)

xls4 = TABLES / "Table_4_stakeholder_corroboration.xlsx"
wb = xlsxwriter.Workbook(str(xls4)); ws = wb.add_worksheet("Table 4")
hdr  = wb.add_format({"bold": True, "bg_color": "#404040", "font_color": "white",
                      "border": 1, "align": "center", "valign": "vcenter", "text_wrap": True})
cell = wb.add_format({"border": 1, "valign": "vcenter", "font_size": 9})
for j, c in enumerate(tab4.columns):
    ws.write(0, j, c, hdr)
for i, (_, r) in enumerate(tab4.iterrows(), start=1):
    for j, c in enumerate(tab4.columns):
        ws.write(i, j, "" if pd.isna(r[c]) else r[c], cell)
ws.set_column(0, 1, 12); ws.set_column(2, 2, 8); ws.set_column(3, 14, 11); ws.set_row(0, 28)
wb.close()
print("Table 4:", len(tab4), "rows ->", csv4.name, "+", xls4.name)
display(tab4.head(8))

In [ ]:
# ── Supp Table 3 — complete corroboration (all 56 cities + 16 clusters + national) ──
_lk = pd.read_csv(RESULTS_AOI / "city_cluster25_lookup.csv")
_city_cl = {str(r["city_name"]).strip(): int(r["Cluster25"]) for _, r in _lk.iterrows()}

def _uc_trend(stats_df, period):
    s = stats_df[(stats_df["zone"] == "urban_core") & (stats_df["period"] == period)]
    if s.empty: return ""
    return "Significant increase" if "increas" in str(s["mk_trend"].iloc[0]).lower() else "No significant trend"

# all-city rows with UC day/night trend from per-city annual stats
_city_rows = []
for city in STUDY_CITIES:
    slug = zone_results[city]["city_slug"]
    ap = RESULTS_03 / slug / f"{slug}_annual_lst.csv"
    if not ap.exists():
        continue
    a = pd.read_csv(ap)
    cid = _city_cl.get(city)
    _city_rows.append({"Level": "City", "Unit": city,
                       "Cluster25": (f"C{cid:02d}" if cid else "C*"),
                       "LST_Day_Trend": _uc_trend(a, "day"), "LST_Night_Trend": _uc_trend(a, "night")})
_allcity = pd.DataFrame(_city_rows)

# merge represented stakeholder columns onto the full city list
_repr_cols = [c for c in comp.columns if c not in ("Level", "LST_Day_Trend", "LST_Night_Trend", "Cluster25")]
_full_city = _allcity.merge(comp[comp["Level"] == "City"][_repr_cols], on="Unit", how="left")
_full_city["N"] = _full_city["N"].fillna(0).astype(int)

# all 16 clusters with cluster UC trend; represented ones carry stakeholder data
_ast = pd.read_csv(STATS03 / "Cluster25_Annual_Stats.csv")
_cl_rows = []
for cid in active_clusters:
    a = _ast[_ast["city"] == f"Cluster25_{cid:02d}"]
    _cl_rows.append({"Level": "Cluster", "Unit": f"Cluster {cid}", "Cluster25": f"C{cid:02d}",
                     "LST_Day_Trend": _uc_trend(a, "day"), "LST_Night_Trend": _uc_trend(a, "night")})
_allcl = pd.DataFrame(_cl_rows)
_repr_cl = comp[comp["Level"] == "Cluster"].copy()
_repr_cl["Cluster25"] = _repr_cl["Cluster25"].str.replace("C0", "C", regex=False)
_allcl["_k"] = _allcl["Cluster25"].str.replace("C0", "C", regex=False)
_full_cl = _allcl.merge(_repr_cl[[c for c in comp.columns if c not in ("Level", "Unit", "LST_Day_Trend", "LST_Night_Trend")]]
                        .assign(_k=lambda d: d["Cluster25"].str.replace("C0", "C", regex=False)).drop(columns="Cluster25"),
                        on="_k", how="left").drop(columns="_k")
_full_cl["N"] = _full_cl["N"].fillna(0).astype(int)

supp3 = pd.concat([_full_city, _full_cl, comp[comp["Level"] == "National"]], ignore_index=True)
supp3 = supp3[[c for c in tab4.columns if c in supp3.columns]]
csvS3 = TABLES / "Supp_Table_3_corroboration_complete.csv"
supp3.to_csv(csvS3, index=False)
print(f"Supp Table 3: {len(supp3)} rows ({(supp3['N']==0).sum()} without stakeholders) -> {csvS3.name}")
display(supp3[supp3["N"] == 0].head(6))

## §2.3 · Short-term PHEs & event-scale SUHII — Figures 10–13, Supp 7–9

Re-rendered / collected from NB04.

- **Figure 10** — daily Urban-Core LST (2024) for **three levels** (Lagos city · Cluster 13 ·
  National), MODIS over ERA5, with the selected-event tier in each legend, night colours in
  black / dark-grey, tightened MODIS↔ERA5 spacing, reduced y-scale, and **no titles** (caption in
  the main text). The **national** level lives inside this panel, so **Supp Fig 7.3 is dropped** —
  supplementary keeps only **7.1 (all cities)** and **7.2 (all clusters)**, in their original format.

In [ ]:
import matplotlib.dates as _mdates
from matplotlib.gridspec import GridSpecFromSubplotSpec as _GSS
from daily_analysis_utils import run_city_event_pipeline

RESULTS_04 = ROOT / "Results" / "04_temporal_analysis_daily"
RAW04 = RESULTS_04 / "raw_extractions"; CLC04 = RESULTS_04 / "cluster_composites"
_YR = STUDY_YEAR
_TIER_C = {"EXCELLENT": "#1a9850", "GOOD": "#f77f00", "FAIR": "#999999", "ACCEPTABLE": "#999999"}

def _period_df(comp, period):
    col = f"{period}_lst_median"
    if comp is None or col not in comp.columns:
        return None
    d = comp[["date", col]].rename(columns={col: "lst_median"}).copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    return d.dropna(subset=["date", "lst_median"]).sort_values("date").reset_index(drop=True)

def _audits(mdf, edf, name):
    return (run_city_event_pipeline(_period_df(mdf, "day"),   _period_df(edf, "day"),   name, "day"),
            run_city_event_pipeline(_period_df(mdf, "night"), _period_df(edf, "night"), name, "night"))

def _national_daily(ds):
    frames = []
    for p in RAW04.glob(f"*_{ds}_UC_daily_{_YR}.csv"):
        d = pd.read_csv(p); d["date"] = pd.to_datetime(d["date"], errors="coerce"); frames.append(d)
    allc = pd.concat(frames)
    num = [c for c in allc.columns if c.startswith(("day_lst", "night_lst"))]
    return allc.groupby("date", as_index=False)[num].mean()

def _daily_panel(ax, df, ds_label, ad, an, ds_type, level_lbl=None, fixed_ylim=None):
    DAY_C, DAY_F = "#d73027", "#fdae61"; NGT_C, NGT_F = "#1C1C1C", "#9a9a9a"
    SEL_D, SEL_N = "#a50026", "#000000"
    if df is None:
        ax.text(0.5, 0.5, f"No {ds_label}", transform=ax.transAxes, ha="center", va="center", color="grey"); return
    df = df.copy(); df["date"] = pd.to_datetime(df["date"]); df = df.sort_values("date")
    if {"day_lst_min", "day_lst_max"} <= set(df.columns):
        ax.fill_between(df["date"], df["day_lst_min"], df["day_lst_max"], color=DAY_F, alpha=.35, lw=0, label="Day range")
    ax.plot(df["date"], df["day_lst_median"], color=DAY_C, lw=1.0, label="Day median", zorder=3)
    if {"night_lst_min", "night_lst_max"} <= set(df.columns):
        ax.fill_between(df["date"], df["night_lst_min"], df["night_lst_max"], color=NGT_F, alpha=.45, lw=0, label="Night range")
    ax.plot(df["date"], df["night_lst_median"], color=NGT_C, lw=1.0, label="Night median", zorder=3)
    key = f"{ds_type}_p90_celsius"
    td = ad.get("thresholds", {}).get(key); tn = an.get("thresholds", {}).get(key)
    if td is not None:
        ax.axhline(td, color=DAY_C, ls="--", lw=.9, alpha=.7, label=f"Day P90 {td:.1f}°", zorder=4)
        h = df[df["day_lst_median"] >= td]; ax.scatter(h["date"], h["day_lst_median"], color=DAY_C, s=9, zorder=5, alpha=.6)
    if tn is not None:
        ax.axhline(tn, color=NGT_C, ls="--", lw=.9, alpha=.7, label=f"Night P90 {tn:.1f}°", zorder=4)
        h = df[df["night_lst_median"] >= tn]; ax.scatter(h["date"], h["night_lst_median"], color=NGT_C, s=9, zorder=5, alpha=.6)
    seen = set()
    for e in ad.get(f"{ds_type}_persistent_events", []) + an.get(f"{ds_type}_persistent_events", []):
        k = (e.get("start_date"), e.get("end_date"))
        if k in seen: continue
        seen.add(k)
        try:
            ax.axvspan(pd.to_datetime(e["start_date"]), pd.to_datetime(e["end_date"]) + pd.Timedelta(days=1),
                       color="grey", alpha=.12, lw=0, zorder=1)
        except Exception: pass
    # ── fix the y-scale FIRST so the selection bracket sits correctly ──
    if fixed_ylim is not None:
        ax.set_ylim(*fixed_ylim)
    else:
        lo = float(np.nanmin(df["night_lst_min"] if "night_lst_min" in df else df["night_lst_median"]))
        hi = float(np.nanmax(df["day_lst_max"] if "day_lst_max" in df else df["day_lst_median"]))
        ax.set_ylim(lo - 2, hi + 2)
    y0, y1 = ax.get_ylim(); gap = (y1 - y0) * .09
    def _br(sd, ed, col, above, tier=None):
        """Draw the selected-event bracket (⊢——⊣) and annotate its tier below it."""
        try: s, e = pd.to_datetime(sd), pd.to_datetime(ed)
        except Exception: return
        bh = (y1 - y0) * .025; b = above - gap
        ax.plot([s, e], [b, b], color=col, lw=2.0, zorder=8)
        ax.plot([s, s], [b - bh, b + bh], color=col, lw=2.0, zorder=9); ax.plot([e, e], [b - bh, b + bh], color=col, lw=2.0, zorder=9)
        if tier:
            ax.annotate(tier, xy=(s + (e - s) / 2, b - bh), xytext=(0, -1.5),
                        textcoords="offset points", ha="center", va="top",
                        fontsize=4.8, fontweight="bold", color=_TIER_C.get(tier, col), zorder=10)
    sd, sn = ad.get("selected_event"), an.get("selected_event")
    if sd and td is not None: _br(sd["start_date"], sd["end_date"], SEL_D, td, sd.get("tier", "FAIR"))
    if sn and tn is not None: _br(sn["start_date"], sn["end_date"], SEL_N, tn, sn.get("tier", "FAIR"))
    ax.set_ylabel((f"{level_lbl}\n" if level_lbl else "") + f"{ds_label} (°C)", fontsize=6.5)
    ax.tick_params(labelsize=5.5); ax.grid(axis="y", ls=":", lw=.4, alpha=.5)
    # legend — no per-tier entries (tier is annotated under each selection bracket)
    hnd, lab = ax.get_legend_handles_labels()
    ax.legend(hnd, lab, loc="upper right", fontsize=5.0, ncol=3, framealpha=.85,
              handlelength=1.1, columnspacing=.7, borderaxespad=.2)

_LG_M = pd.read_csv(RAW04 / f"Lagos_MODIS_UC_daily_{_YR}.csv"); _LG_E = pd.read_csv(RAW04 / f"Lagos_ERA5_UC_daily_{_YR}.csv")
_C13_M = pd.read_csv(CLC04 / f"Cluster13_MODIS_composite_{_YR}.csv"); _C13_E = pd.read_csv(CLC04 / f"Cluster13_ERA5_composite_{_YR}.csv")
_NAT_M = _national_daily("MODIS"); _NAT_E = _national_daily("ERA5")
_LEVELS = [("Lagos (city)", _LG_M, _LG_E), ("Cluster 13", _C13_M, _C13_E), ("National", _NAT_M, _NAT_E)]

# ── uniform temperature scale across ALL six sub-panels ──
def _series_range(d):
    lo = float(np.nanmin(d["night_lst_min"] if "night_lst_min" in d else d["night_lst_median"]))
    hi = float(np.nanmax(d["day_lst_max"] if "day_lst_max" in d else d["day_lst_median"]))
    return lo, hi
_ranges = [_series_range(d) for d in (_LG_M, _LG_E, _C13_M, _C13_E, _NAT_M, _NAT_E) if d is not None]
_GYLIM = (min(r[0] for r in _ranges) - 2, max(r[1] for r in _ranges) + 2)

fig10 = plt.figure(figsize=(8.27, 9.6))
_outer = fig10.add_gridspec(3, 1, hspace=0.14, top=0.985, bottom=0.05, left=0.10, right=0.97)
for _i, (_lbl, _m, _e) in enumerate(_LEVELS):
    _ad, _an = _audits(_m, _e, _lbl)
    _in = _GSS(2, 1, subplot_spec=_outer[_i], hspace=0.06)
    _axm = fig10.add_subplot(_in[0]); _axe = fig10.add_subplot(_in[1], sharex=_axm)
    _daily_panel(_axm, _m, "MODIS", _ad, _an, "modis", level_lbl=_lbl, fixed_ylim=_GYLIM)
    _daily_panel(_axe, _e, "ERA5-Land", _ad, _an, "era5", fixed_ylim=_GYLIM)
    _axm.tick_params(labelbottom=False)
    _axe.xaxis.set_major_locator(_mdates.MonthLocator()); _axe.xaxis.set_major_formatter(_mdates.DateFormatter("%b"))
    _axe.tick_params(axis="x", labelsize=5.5)
paths = savefig(fig10, MAIN_FIG / "Figure_10_daily_phe_panel")
plt.show(); print("Figure 10:", [p.name for p in paths], "| uniform y =", tuple(round(v,1) for v in _GYLIM))

In [ ]:
# ── Supp 7.1 (all cities) / 7.2 (all clusters) — collected daily charts (original format) ──
SUP7 = SUP_FIG / "7_daily_phe"; SUP7.mkdir(exist_ok=True)
S71 = SUP7 / "7.1_city"; S71.mkdir(exist_ok=True); S72 = SUP7 / "7.2_cluster"; S72.mkdir(exist_ok=True)
_CDT = RESULTS_04 / "charts" / "city_daily_timeseries"
_CLD = RESULTS_04 / "charts" / "cluster_daily_composites"
_n71 = 0
for i, city in enumerate(STUDY_CITIES, start=1):
    slug = zone_results[city]["city_slug"]
    for ext in ("svg", "pdf"):
        # NB04 named a few multi-word cities with spaces rather than underscores
        for _base in (slug, slug.replace("_", " ")):
            s = _CDT / f"{_base}_daily_LST_{_YR}.{ext}"
            if s.exists():
                shutil.copyfile(s, S71 / f"Supp_Fig_7.1.{i}_{slug}.{ext}"); _n71 += (ext == "svg"); break
_n72 = 0
for i, cid in enumerate(active_clusters, start=1):
    for ext in ("svg", "pdf"):
        s = _CLD / f"Cluster{cid:02d}_daily_LST_{_YR}.{ext}"
        if s.exists(): shutil.copyfile(s, S72 / f"Supp_Fig_7.2.{i}_C{cid}.{ext}"); _n72 += (ext == "svg")
print(f"Supp 7.1 collected {_n71} city charts; Supp 7.2 collected {_n72} cluster charts (original format)")

### Figure 11, Figure 12, Figure 13 · Supp Fig 8 / 9 · Supp Tables 4 & 5

- **Figure 11** — cluster ERA5 heat-event bars (re-rendered): one shared legend on a single bottom
  row with its descriptive caption as the legend title; per-panel subtitles retained; main title removed.
- **Figure 12** — target diagram (MODIS vs ERA5-Land), collected (vector).
- **Figure 13** — city-level SEVIRI SUHII (re-rendered): cluster (fill) legend widened to **4 columns**.
- **Supp Fig 8.1 / 8.2** — cluster event-frequency, day / night (collected).
- **Supp Fig 9** — ERA5-Land vs MODIS pooled hexbin (re-rendered): title uses a plain hyphen.
- **Supp Table 4** — event-frequency breakdown across city / cluster / national levels (ERA5 PHE basis).
- **Supp Table 5** — per-city selected-event windows, tiers and MODIS↔ERA5 correlations.

In [ ]:
# ── Figure 12 (collect) + Supp Fig 8.1 / 8.2 (collect) ──
_CORR = RESULTS_04 / "correlation"; _CHARTS = RESULTS_04 / "charts"
for label, s, dst, base in [
    ("Figure 12",  _CORR / "target_diagram_MODIS_vs_ERA5-Land.svg", MAIN_FIG, "Figure_12_target_diagram"),
    ("Supp Fig 8.1", _CHARTS / "Cluster_event_frequency_day.svg",   SUP_FIG / "8_event_frequency", "Supp_Fig_8.1_cluster_event_frequency_day"),
    ("Supp Fig 8.2", _CHARTS / "Cluster_event_frequency_night.svg", SUP_FIG / "8_event_frequency", "Supp_Fig_8.2_cluster_event_frequency_night"),
]:
    dst.mkdir(exist_ok=True)
    picked = []
    for ext in ("svg", "pdf"):
        sf = s.with_suffix(f".{ext}")
        if sf.exists(): shutil.copyfile(sf, dst / f"{base}.{ext}"); picked.append(ext)
    print(f"{label:12} <- {s.name:44} [{', '.join(picked) or 'MISSING'}]")

# ── Supp Table 4 — event-frequency breakdown (city / cluster / national; ERA5 PHE basis) ──
_ce = pd.read_csv(RESULTS_04 / "city_heat_event_summary.csv")
_stats = ["n_persistent_events", "total_persistent_days", "longest_event_days", "mean_event_duration_days"]
_ecols = [f"era5_{p}_{s}" for p in ("day", "night") for s in _stats]
def _t4_row(level, entity, cl, row):
    d = {"Level": level, "Entity": entity, "Cluster25": (f"C{int(cl)}" if pd.notna(cl) else "")}
    for p in ("day", "night"):
        for s, nm in zip(_stats, ["Events", "PersistDays", "Longest", "MeanDur"]):
            d[f"{p.title()}_{nm}"] = round(float(row[f"era5_{p}_{s}"]), 1)
    return d
_rows4 = [_t4_row("City", r["city"], r["cluster25"], r) for _, r in _ce.iterrows()]
for cl, g in _ce.groupby("cluster25"):
    _rows4.append(_t4_row("Cluster", f"Cluster {int(cl)}", cl, g[_ecols].mean()))
_rows4.append(_t4_row("National", "National", float("nan"), _ce[_ecols].mean()))
supp4 = pd.DataFrame(_rows4)
csvS4 = TABLES / "Supp_Table_4_event_frequency.csv"; supp4.to_csv(csvS4, index=False)

# ── Supp Table 5 — selected-event windows, tiers, correlations (per city) ──
_ren = {"city": "City", "cluster25": "Cluster25", "validation_status": "Overall_Tier",
        "day_sel_start": "Day_Start", "day_sel_end": "Day_End", "day_sel_duration_days": "Day_Days", "day_sel_tier": "Day_Tier",
        "day_corr_r": "Day_r", "day_corr_r2": "Day_r2", "day_corr_bias": "Day_Bias", "day_corr_rmse": "Day_RMSE",
        "night_sel_start": "Night_Start", "night_sel_end": "Night_End", "night_sel_duration_days": "Night_Days", "night_sel_tier": "Night_Tier",
        "night_corr_r": "Night_r", "night_corr_r2": "Night_r2", "night_corr_bias": "Night_Bias", "night_corr_rmse": "Night_RMSE"}
supp5 = _ce[[c for c in _ren if c in _ce.columns]].rename(columns=_ren)
supp5["Cluster25"] = supp5["Cluster25"].map(lambda c: f"C{int(c)}")
csvS5 = TABLES / "Supp_Table_5_selected_events_correlations.csv"; supp5.to_csv(csvS5, index=False)
print(f"Supp Table 4: {len(supp4)} rows -> {csvS4.name}")
print(f"Supp Table 5: {len(supp5)} rows -> {csvS5.name}")
display(supp4.head(4)); display(supp5.head(4))

In [ ]:
# ── Figure 11 — cluster ERA5 heat-event bars ─────────────────────────────────
# Re-render of NB04 `heat_event_bars_combined_composite`: one shared legend on a
# single bottom row (caption becomes the legend title); subtitles kept; no suptitle.
import matplotlib.patches as _mpatch
_clh = pd.read_csv(RESULTS_04 / "cluster_heat_event_summary.csv")
_TIER_EC = {"EXCELLENT": "#1a9850", "GOOD": "#f77f00", "FAIR": "#999999"}
_DAY_C, _NGT_C = "#d73027", "#4575b4"

def _bar_group(ax, df_cl, day_col, ngt_col, ylabel, title, bw=0.35):
    _sv = (df_cl.dropna(subset=["cluster25"]).set_index("cluster25")
           [[f"{day_col}_mean", f"{ngt_col}_mean"]].fillna(0)
           .max(axis=1).sort_values(ascending=False))
    cids = _sv.index.tolist(); xs = np.arange(len(cids))
    for xi, cid in enumerate(cids):
        row = df_cl[df_cl["cluster25"] == cid].iloc[0]
        ec  = _TIER_EC.get(str(row.get("modal_validation_tier", "")), "#cccccc")
        for col, xoff, cc in [(day_col, -bw / 2, _DAY_C), (ngt_col, +bw / 2, _NGT_C)]:
            m  = float(row.get(f"{col}_mean", 0) or 0)
            lo = float(row.get(f"{col}_min", m) or m)
            hi = float(row.get(f"{col}_max", m) or m)
            ax.bar(xi + xoff, m, width=bw, color=cc, alpha=0.75,
                   edgecolor=ec, linewidth=1.4, zorder=3)
            ax.errorbar(xi + xoff, m, yerr=[[max(m - lo, 0)], [max(hi - m, 0)]],
                        fmt="none", ecolor="#333333", elinewidth=1.0, capsize=2.5, zorder=4)
    ax.set_xticks(xs)
    ax.set_xticklabels([f"C{int(c):02d}" for c in cids], fontsize=7, rotation=45, ha="right")
    ax.set_xlabel("Cluster", fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title, fontsize=9, pad=4)
    ax.grid(axis="y", linestyle=":", alpha=0.45)
    ax.spines[["top", "right"]].set_visible(False)

fig11, (a11, b11) = plt.subplots(1, 2, figsize=(11.69, 5.85), gridspec_kw={"wspace": 0.28})
_bar_group(a11, _clh, "day_n_era5_persist_evts", "night_n_era5_persist_evts", "Events", "ERA5 Persistent Events")
_bar_group(b11, _clh, "day_era5_longest_days", "night_era5_longest_days", "Days", "ERA5 Longest Event")
_leg11 = [
    _mpatch.Patch(color=_DAY_C, alpha=0.75, label="Daytime"),
    _mpatch.Patch(color=_NGT_C, alpha=0.75, label="Night-time"),
    _mpatch.Patch(fc="w", ec="#1a9850", lw=1.5, label="EXCELLENT tier"),
    _mpatch.Patch(fc="w", ec="#f77f00", lw=1.5, label="GOOD tier"),
    _mpatch.Patch(fc="w", ec="#999999", lw=1.5, label="FAIR tier"),
]
# Drop the legend ~1 cm below the plots (clear of the rotated tick labels) with a
# "Legend" header, and keep the descriptive caption as a thin line beneath the box.
fig11.tight_layout(rect=(0, 0.06, 1, 1))
fig11.legend(handles=_leg11, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.017),
             fontsize=8, framealpha=0.9, columnspacing=1.4, handlelength=1.5,
             title="Legend", title_fontsize=9, alignment="center")
fig11.text(0.5, -0.135,
           "Bars = cluster mean across cities  ·  Error bars = city-level min–max  ·  Bar border = validation tier",
           ha="center", va="top", fontsize=7, color="#555", style="italic")
_p11 = savefig(fig11, MAIN_FIG / "Figure_11_heat_event_bars")
plt.show()
print("Figure 11 ->", ", ".join(p.name for p in _p11))

In [ ]:
# ── Figure 13 — city-level SEVIRI SUHII (cluster-fill legend widened to 4 columns) ──
from src.visualization import create_suhii_city_chart
_seviri13 = pd.read_csv(RESULTS_04 / "seviri" / "City_SEVIRI_3zone_SUHII.csv")
_ce13     = pd.read_csv(RESULTS_04 / "city_heat_event_summary.csv")
_clu13    = {r["city"]: int(r["cluster25"]) for _, r in _ce13.iterrows() if pd.notna(r["cluster25"])}
fig13 = create_suhii_city_chart(
    df_seviri      = _seviri13,
    cluster_lookup = _clu13,
    output_dir     = str(MAIN_FIG),
    output_stem    = "Figure_13_seviri_suhii_cities",
    dpi            = config.FIGURE_DPI,
    cl_legend_ncol = 4,
)
if fig13 is not None:
    plt.show()
    print("Figure 13 -> Figure_13_seviri_suhii_cities.(svg|pdf)  [cluster legend = 4 columns]")
else:
    print("Figure 13 — no valid SEVIRI SUHII rows")

In [ ]:
# ── Supp Fig 9 — ERA5-Land vs MODIS daily hexbin, all cities pooled (title uses '-') ──
from src.visualization import create_era5_modis_hexbin
from src.daily_analysis_utils import load_daily_csv, compute_correlation_metrics
_RAW09 = RESULTS_04 / "raw_extractions"
_cities09 = pd.read_csv(RESULTS_04 / "city_heat_event_summary.csv")["city"].tolist()
_pooled09 = {"day": {"era5": [], "modis": []}, "night": {"era5": [], "modis": []}}
for _city in _cities09:
    _mcsv = _RAW09 / f"{_city}_MODIS_UC_daily_{STUDY_YEAR}.csv"
    _ecsv = _RAW09 / f"{_city}_ERA5_UC_daily_{STUDY_YEAR}.csv"
    if not (_mcsv.exists() and _ecsv.exists()):
        continue
    for _p in ("day", "night"):
        _m = load_daily_csv(_mcsv, _p); _e = load_daily_csv(_ecsv, _p)
        if _m is None or _e is None:
            continue
        _mg = pd.merge(
            _m[["date", "lst_median"]].rename(columns={"lst_median": "modis"}),
            _e[["date", "lst_median"]].rename(columns={"lst_median": "era5"}),
            on="date", how="inner").dropna()
        if _mg.empty:
            continue
        _pooled09[_p]["era5"]  += _mg["era5"].tolist()
        _pooled09[_p]["modis"] += _mg["modis"].tolist()
for _p in ("day", "night"):
    _pooled09[_p]["metrics"] = compute_correlation_metrics(
        pd.Series(_pooled09[_p]["modis"]), pd.Series(_pooled09[_p]["era5"]))
_S9DIR = SUP_FIG / "9_hexbin"; _S9DIR.mkdir(exist_ok=True)
fig9 = create_era5_modis_hexbin(
    paired_data = _pooled09,
    title       = f"All Cities Pooled - ERA5-Land vs MODIS ({STUDY_YEAR})",
    output_path = str(_S9DIR / "Supp_Fig_9_era5_vs_modis_hexbin.svg"),
    dpi         = config.FIGURE_DPI,
    show_inline = True,
)
if fig9 is not None:      # emit a PDF sibling (function itself writes only the SVG)
    fig9.savefig(str(_S9DIR / "Supp_Fig_9_era5_vs_modis_hexbin.pdf"),
                 format="pdf", bbox_inches="tight", facecolor="white")
print("Supp Fig 9 -> Supp_Fig_9_era5_vs_modis_hexbin.(svg|pdf)  "
      f"[day n={_pooled09['day']['metrics'].get('n')}, night n={_pooled09['night']['metrics'].get('n')}]")

## §2.4 · Surface hot- and cold-spot detection & delineation

*Manuscript §3.3.* Gi\* hotspot detection during the selected PHEs, the resulting hot/cold-spot
zone footprints (L50/L75/L90), and stakeholder corroboration. Assembled from cached **NB05**
outputs (`Results/05_hotspot_analysis/`).

| Item | Content | Handling |
|---|---|---|
| **Figure 14** | Gi\* Z-score distribution, Uyo (C13), day + night | collected |
| **Supp Fig 10.1–10.55** | Gi\* Z-score distribution, all other cities | collected (as-is) |
| **Table 5** | Cluster-level significant-spot occurrence (Day/Night × Hot/Cool) | csv + styled xlsx |
| **Supp Table 6** | City-level zone footprints (n, L50, L75, L90, BH) | csv + styled xlsx |
| **Figure 15 / 16** | Spatial extent of spots, per city — day / night | re-rendered (§2.4b) |
| **Figure 17** | Hot/cold zone-area distribution across clusters (4-panel) | re-rendered (§2.4b) |
| **Supp Fig 11.1 / 11.2** | City hot-/cold-spot area ranked | collected |
| **Figure 18** | Stakeholder buffer-sensitivity corroboration (national) | collected |
| **Supp Table 7** | Stakeholder buffer-sensitivity hit-rate (%) by radius | csv + styled xlsx |

In [ ]:
RESULTS_05  = ROOT / "Results" / "05_hotspot_analysis"
RESULTS_06T = ROOT / "Results" / "06_hotspot_modelling" / "tables"
_F05 = RESULTS_05 / "figures"

# ── Figure 14 — Uyo Gi* Z-score distribution ──
_S10 = SUP_FIG / "10_gistar_distributions"; _S10.mkdir(exist_ok=True)
def _copy_pair(stem_src, dst_dir, stem_dst):
    picked = []
    for ext in ("svg", "pdf"):
        s = _F05 / f"{stem_src}.{ext}"
        if s.exists():
            shutil.copyfile(s, dst_dir / f"{stem_dst}.{ext}"); picked.append(ext)
    return picked

p14 = _copy_pair("Uyo_gistar_dist", MAIN_FIG, "Figure_14_uyo_gistar_dist")
print("Figure 14 (Uyo) ->", p14 or "MISSING")

# ── Supp Fig 10.1–10.55 — all other cities' Gi* distributions (alphabetical, Uyo excluded) ──
_gis = sorted(p.stem.replace("_gistar_dist", "") for p in _F05.glob("*_gistar_dist.svg"))
_others = [c for c in _gis if c != "Uyo"]
_n10 = 0
for i, city in enumerate(_others, start=1):
    got = _copy_pair(f"{city}_gistar_dist", _S10, f"Supp_Fig_10.{i}_{city.replace(' ', '_')}")
    _n10 += ("svg" in got)
print(f"Supp Fig 10: collected {_n10} city Gi* distributions (of {len(_others)} non-Uyo cities)")

In [ ]:
# ── Supp Fig 11.1 / 11.2 — city hot-/cold-spot area ranked (collected) ──
# NOTE: mapped 11.1 = hotspot, 11.2 = coldspot (per "Hotspot and Coldspot"); the file list
# in the brief had coldspot first — trivially swappable if the other order was intended.
_S11 = SUP_FIG / "11_city_area_ranked"; _S11.mkdir(exist_ok=True)
for src, dst in [("city_hotspot_area_ranked",  "Supp_Fig_11.1_city_hotspot_area_ranked"),
                 ("city_coldspot_area_ranked", "Supp_Fig_11.2_city_coldspot_area_ranked")]:
    print(f"{dst:44} <- {src:30} [{', '.join(_copy_pair(src, _S11, dst)) or 'MISSING'}]")

# ── Figure 18 — stakeholder buffer-sensitivity corroboration (national) ──
p18 = _copy_pair("stakeholder_hitrate_national", MAIN_FIG, "Figure_18_stakeholder_hitrate_national")
print("Figure 18 ->", p18 or "MISSING")

In [ ]:
# ── Tables (§3.3 + §3.4): Table 5, Supp 6, Supp 7, Table 6, Table 7, Supp 8 ──
# All six are built together from cached NB05/NB06 CSVs into manuscript layouts
# (plain .csv + styled .xlsx) by src/pub_hotspot_tables.py.
import pub_hotspot_tables as _pht
import importlib as _il; _il.reload(_pht)
_tbls = _pht.build_hotspot_tables(RESULTS_05, RESULTS_06T, TABLES)
for _k, _v in _tbls.items():
    print(f"  {_k:14} {_v.shape[0]:>3} rows x {_v.shape[1]} cols")
print("\nTable 5 (cluster spot occurrence) — preview:")
display(_tbls["Table_5"].head(4))
print("Supp Table 7 (buffer sensitivity) — preview:")
display(_tbls["Supp_Table_7"].drop(columns=["_group"]).head(3))

### Figure 15 / 16 · Per-city spatial extent of hot/cold spots (day / night)

The two A4 city-grid maps are **re-rendered by post-processing the cached NB05 SVGs** (the
underlying per-city LST rasters are not re-computed): the two-line figure title is removed, the
north arrow is moved to sit **beside the LST colour bar**, and the shared map legend (§11a2) is
composited into a bottom strip — so each figure (Figure 15 = day, Figure 16 = night) is
self-contained with its own legend and arrow, matching the manuscript.

In [ ]:
from lxml import etree as _ET
import fitz as _fitz
_SVGNS = "http://www.w3.org/2000/svg"
_ECOMM = _ET.Comment("").__class__

def _gcomment(g):
    for ch in g:
        if isinstance(ch, _ECOMM):
            return (ch.text or "").strip()
    return ""

def _fig1516(src, dst_svg):
    tree = _ET.parse(str(src)); root = tree.getroot()
    _vb = (root.get("viewBox") or "0 0 575.30448 864.869096").split()
    W, H = float(_vb[2]), float(_vb[3])
    _title = ("Urban Heat Island", "KDE 50% contours")
    drop, arrow = [], None
    for g in root.iter(f"{{{_SVGNS}}}g"):
        gid = g.get("id", ""); c = _gcomment(g)
        if gid.startswith("text_") and any(m in c for m in _title):
            drop.append(g)
        if gid.startswith("axes_"):
            _txts = [ch for ch in g.iter(f"{{{_SVGNS}}}g") if ch.get("id", "").startswith("text_")]
            if _txts and all(_gcomment(t) == "N" for t in _txts):
                arrow = g
    for g in drop:
        g.getparent().remove(g)
    if arrow is not None:                       # move N-arrow down to the colour-bar level
        dy = ((1 - 0.021) - (1 - 0.972)) * H
        arrow.set("transform", f"translate(-18.000,{dy:.3f}) {arrow.get('transform', '')}".strip())
    # composite the shared map legend into a new bottom strip
    _legW, _legH = 582.559375, 64.8
    root.set("height", f"{H + _legH}pt"); root.set("viewBox", f"0 0 {W} {H + _legH}")
    _defs = root.find(f"{{{_SVGNS}}}defs")
    _lr = _ET.parse(str(_F05 / "a4_grid_legend.svg")).getroot()
    _ld = _lr.find(f"{{{_SVGNS}}}defs")
    _have = {c.get("id") for c in _defs if c.get("id")}
    if _ld is not None:
        for c in list(_ld):
            if not (c.get("id") and c.get("id") in _have):
                _defs.append(c)
    _lf = _lr.find(f"{{{_SVGNS}}}g[@id='figure_1']"); _lf.set("id", "legend_fig")
    _wrap = _ET.SubElement(root, f"{{{_SVGNS}}}g")
    _wrap.set("transform", f"translate({(W - _legW) / 2.0:.3f},{H:.3f})"); _wrap.append(_lf)
    dst_svg.parent.mkdir(parents=True, exist_ok=True)
    tree.write(str(dst_svg), xml_declaration=True, encoding="utf-8")
    dst_svg.with_suffix(".pdf").write_bytes(_fitz.open(str(dst_svg)).convert_to_pdf())

for _per, _fn in [("day", "15"), ("night", "16")]:
    _src = _F05 / f"all_cities_hotspot_A4_{_per}.svg"
    _dst = MAIN_FIG / f"Figure_{_fn}_spatial_extent_{_per}.svg"
    if _src.exists():
        _fig1516(_src, _dst); print(f"Figure {_fn} ({_per}) -> {_dst.name} (+pdf)")
    else:
        print(f"Figure {_fn}: MISSING {_src.name}")

### Figure 17 · Hot/cold zone-area distribution across clusters (2×2)

Re-rendered from cached NB05 zone footprints (`spot_comprehensive_summary.csv`) into the
manuscript's four-panel layout — **Hotspot Day** (top-left) · **Hotspot Night** (top-right) ·
**Cold spot Day** (bottom-left) · **Cold spot Night** (bottom-right). Bars are coloured by
Cluster25 with the nested KDE zones stacked (L90 dense core → L75 inner → L50 outer extent).
The zone key and the Cluster25 key are placed **at the bottom, side by side** — the per-panel
zonal-area inset boxes are removed.

In [ ]:
from matplotlib.patches import Patch as _Patch17
_CLR = config.CLUSTER25_COLORS
_sc17 = pd.read_csv(RESULTS_05 / "spot_comprehensive_summary.csv")
_g17 = (_sc17.groupby(["Type", "Period", "Cluster"])
        .agg(L50=("area_50_km2", "sum"), L75=("area_75_km2", "sum"), L90=("area_90_km2", "sum"))
        .reset_index())
_PANELS17 = [("hot", "day", "Hotspot — Day"), ("hot", "night", "Hotspot — Night"),
             ("cold", "day", "Cold spot — Day"), ("cold", "night", "Cold spot — Night")]
fig17, _axes17 = plt.subplots(2, 2, figsize=(11.69, 8.0))
_allcl17 = set()
for _ax, (_typ, _per, _title) in zip(_axes17.flat, _PANELS17):
    _sub = _g17[(_g17.Type == _typ) & (_g17.Period == _per)].copy()
    _sub = _sub[_sub["L50"] > 0].sort_values("L50", ascending=False)
    if _sub.empty:
        _ax.set_visible(False); continue
    _x = np.arange(len(_sub))
    for _xi, _r in zip(_x, _sub.itertuples()):
        _cl = int(_r.Cluster); _col = _CLR.get(_cl, "#888888"); _allcl17.add(_cl)
        _l90, _l75, _l50 = float(_r.L90), float(_r.L75), float(_r.L50)
        _ax.bar(_xi, _l90, color=_col, alpha=1.0, hatch="xxx", edgecolor="white", linewidth=0.3, zorder=3)
        _ax.bar(_xi, max(_l75 - _l90, 0), bottom=_l90, color=_col, alpha=0.72, hatch="//", edgecolor="white", linewidth=0.3, zorder=3)
        _ax.bar(_xi, max(_l50 - _l75, 0), bottom=_l75, color=_col, alpha=0.42, edgecolor="white", linewidth=0.3, zorder=3)
    _ax.set_xticks(_x); _ax.set_xticklabels([f"C{int(c)}" for c in _sub["Cluster"]], rotation=45, ha="right", fontsize=7.5)
    _ax.set_ylabel("Total area (km²)", fontsize=9)
    _ax.set_title(_title, fontsize=11, fontweight="bold", pad=5)
    _ax.grid(axis="y", ls=":", lw=0.4, alpha=0.5); _ax.spines[["top", "right"]].set_visible(False)
fig17.tight_layout(rect=(0, 0.11, 1, 1))
_zh17 = [_Patch17(facecolor="#555", alpha=1.0, hatch="xxx", edgecolor="white", label="L90 dense core"),
         _Patch17(facecolor="#555", alpha=0.72, hatch="//", edgecolor="white", label="L75 inner zone"),
         _Patch17(facecolor="#555", alpha=0.42, edgecolor="white", label="L50 outer extent")]
_lg17a = fig17.legend(handles=_zh17, title="Contour zone", loc="lower left", bbox_to_anchor=(0.02, 0.01),
                      ncol=1, fontsize=8, title_fontsize=9, framealpha=0.9)
_lg17a.get_title().set_fontweight("bold")
_ch17 = [_Patch17(facecolor=_CLR.get(c, "#888"), label=f"C{c}") for c in sorted(_allcl17)]
_lg17b = fig17.legend(handles=_ch17, title="Cluster25", loc="lower center", bbox_to_anchor=(0.60, 0.005),
                      ncol=8, fontsize=7.5, title_fontsize=9, framealpha=0.9, columnspacing=1.0, handlelength=1.2)
_lg17b.get_title().set_fontweight("bold")
_p17 = savefig(fig17, MAIN_FIG / "Figure_17_zone_area_distribution")
plt.show(); print("Figure 17 ->", ", ".join(p.name for p in _p17))

## §2.5 · Association of physiographic typology (PLT) with surface thermal anomaly

*Manuscript §3.4.* Tests of whether the composite PLT typology explains hot/cold-spot occurrence,
extent and intensity, and how much the composite adds beyond its best single element. Assembled
from cached **NB06** outputs (`Results/06_hotspot_modelling/tables/`).

| Item | Content | Handling |
|---|---|---|
| **Table 6** | PLT–thermal association (Occurrence / SUHII / Area × Hot/Cold × Day/Night) | csv + styled xlsx |
| **Figure 19** | Forest plot of effect sizes (composite + element predictors) | re-rendered (§2.5b) |
| **Table 7** | Excerpt — best single-element attribution (composite vs element) | csv + styled xlsx |
| **Supp Table 8** | Full single-element attribution (all 12 outcomes) | csv + styled xlsx |

> **Source note.** Table 7 / Supp Table 8 are built from `master_synthesis.csv` — the complete
> "one major table" that reproduces the manuscript excerpt exactly (it includes the Occurrence
> rows that `synthesis_S2_comparative_power.csv` omits, that file holding only the SUHII/Area η²
> comparison). The six tables above were all produced by the single call in §2.4.

In [ ]:
# Table 6, Table 7 and Supp Table 8 were written by the build in §2.4 — preview them here.
print("Table 6 (PLT–thermal association):")
display(_tbls["Table_6"])
print("Table 7 (best single-element attribution — excerpt):")
display(_tbls["Table_7"])

### Figure 19 · Forest plot of effect sizes

Re-rendered from the cached NB06 attribution tables (`partA_synthesis_cluster25.csv` +
`partB_{hot,cold}_{day,night}.csv`). Same design as NB06 — three outcome panels (Occurrence /
SUHII / Area), effect-size-zone bands, predictor markers and significance outlines — with the
legend band pulled in tight under the plots while keeping the three sections
(**Effect-size zone · Predictor · Significance**) separated.

In [ ]:
from matplotlib.patches import Patch as _Patch, FancyBboxPatch as _FBox
from matplotlib.lines import Line2D as _Ln

_SPOT, _PER = ['hot', 'cold'], ['day', 'night']
_ELEMS = ['climate_class', 'geology_class', 'elevation_class']
_ELEM_NAME = {'climate_class': 'Climate Zone', 'geology_class': 'Geology', 'elevation_class': 'Elevation Belt'}
_OUTLBL = {'occurrence': 'Occurrence', 'suhii': 'SUHII', 'area': 'Area (L50)'}
_ALPHA = 0.05
def _pf(x):
    s = str(x).strip()
    return 0.0005 if s.startswith('<') else (float(s) if s.replace('.', '', 1).replace('-', '', 1).isdigit() else 1.0)
def _stars(p): return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))

_pa = pd.read_csv(RESULTS_06T / 'partA_synthesis_cluster25.csv')
_paSRC = {'occurrence': {}, 'suhii': {}, 'area': {}}
for _, _r in _pa.iterrows():
    _st, _pe = str(_r['Spot Type']).lower(), str(_r['Period']).lower()
    _paSRC['occurrence'][(_st, _pe)] = {'cramers_v': float(_r['V (occ)']), 'p': _pf(_r['p (occ)'])}
    _paSRC['suhii'][(_st, _pe)] = {'eta2': float(_r['η² (suhii)']), 'p': _pf(_r['p (suhii)'])}
    _paSRC['area'][(_st, _pe)] = {'eta2': float(_r['η² (area)']), 'p': _pf(_r['p (area)'])}
_pbR, _pbH = {}, {}
_name2key = {v: k for k, v in _ELEM_NAME.items()}
for _st in _SPOT:
    for _pe in _PER:
        _pb = pd.read_csv(RESULTS_06T / f'partB_{_st}_{_pe}.csv')
        for _, _r in _pb.iterrows():
            _e = _name2key.get(str(_r['Element']).strip())
            if _e is None: continue
            _pbR[(_st, _pe, _e, 'occurrence')] = {'cramers_v': float(_r['V'])}
            _pbR[(_st, _pe, _e, 'suhii')] = {'eta2': float(_r['η²_suhii'])}
            _pbR[(_st, _pe, _e, 'area')] = {'eta2': float(_r['η²_area'])}
            _pbH[(_st, _pe, _e, 'occurrence')] = _pf(_r['p(Holm)_occ'])
            _pbH[(_st, _pe, _e, 'suhii')] = _pf(_r['p(Holm)_suhii'])
            _pbH[(_st, _pe, _e, 'area')] = _pf(_r['p(Holm)_area'])

_TH = {'occurrence': {'s': .10, 'm': .30, 'l': .50, 'fl': .60, 'lbl': "Cramér's V"},
       'suhii': {'s': .01, 'm': .06, 'l': .14, 'fl': .22, 'lbl': 'η²'},
       'area': {'s': .01, 'm': .06, 'l': .14, 'fl': .22, 'lbl': 'η² (L50)'}}
_ZC = {'negligible': '#FFAAAA', 'small': '#FFCC99', 'medium': '#FFEE99', 'large': '#AADDAA'}; _ZA = 0.42
_CFILL = '#A8D8A8'
_EFILL = {'climate_class': '#F4A8A8', 'geology_class': '#A8A8F4', 'elevation_class': '#F4CCA8'}
_EMARK = {'climate_class': 'D', 'geology_class': 's', 'elevation_class': '^'}
_ECS, _ECN, _LWS, _LWN = '#CC1100', '#111111', 2.0, 0.9
_FB, _FT = 10, 11
def _effs(out):
    v = []
    for st in _SPOT:
        for pe in _PER:
            a = _paSRC[out].get((st, pe))
            if a: v.append(a['cramers_v'] if out == 'occurrence' else a['eta2'])
            for e in _ELEMS:
                b = _pbR.get((st, pe, e, out))
                if b: v.append(b['cramers_v'] if out == 'occurrence' else b['eta2'])
    return v

fig19, _axes = plt.subplots(1, 3, figsize=(11.69, 7.6), gridspec_kw={'wspace': 0.46})
for _ax, _out in zip(_axes, ['occurrence', 'suhii', 'area']):
    _t = _TH[_out]; _s, _m, _l = _t['s'], _t['m'], _t['l']
    _al = _effs(_out); _xl = max(_t['fl'], max(_al) * 1.15) if _al else _t['fl']
    _ax.set_title(_OUTLBL[_out], fontsize=_FT, fontweight='bold', pad=8)
    _ax.set_xlabel(_t['lbl'], fontsize=_FB); _ax.set_xlim(-0.01, _xl)
    _ax.axvspan(-0.01, _s, color=_ZC['negligible'], alpha=_ZA, zorder=0)
    _ax.axvspan(_s, _m, color=_ZC['small'], alpha=_ZA, zorder=0)
    _ax.axvspan(_m, _l, color=_ZC['medium'], alpha=_ZA, zorder=0)
    _ax.axvspan(_l, _xl, color=_ZC['large'], alpha=_ZA, zorder=0)
    for _rf, _ls in ((_s, '--'), (_m, ':'), (_l, '-.')):
        _ax.axvline(_rf, color='#888888', lw=0.7, ls=_ls, zorder=1)
    _ax.axvline(0, color='#CCCCCC', lw=0.7, zorder=1)
    _yl, _yp, _yc = [], [], 0
    for st in _SPOT:
        for pe in _PER:
            _yl.append(f'{st.capitalize()} / {pe.capitalize()}'); _yp.append(_yc)
            a = _paSRC[_out].get((st, pe))
            if a:
                _eA = a['cramers_v'] if _out == 'occurrence' else a['eta2']; _sA = a['p'] < _ALPHA
                _ax.scatter(_eA, _yc, color=_CFILL, s=110, marker='o', zorder=5,
                            edgecolors=_ECS if _sA else _ECN, linewidths=_LWS if _sA else _LWN)
                if _sA:
                    _ax.text(_eA + _xl * 0.015, _yc + 0.20, _stars(a['p']), fontsize=8,
                             color=_ECS, va='bottom', fontweight='bold')
            _off = {'climate_class': -0.22, 'geology_class': 0.0, 'elevation_class': 0.22}
            for e in _ELEMS:
                b = _pbR.get((st, pe, e, _out))
                if b:
                    _eB = b['cramers_v'] if _out == 'occurrence' else b['eta2']
                    _sB = _pbH.get((st, pe, e, _out), 1.0) < _ALPHA
                    _ax.scatter(_eB, _yc + _off[e], color=_EFILL[e], s=60, marker=_EMARK[e], zorder=4,
                                edgecolors=_ECS if _sB else _ECN, linewidths=_LWS if _sB else _LWN)
            _yc += 1
    _ax.set_yticks(_yp); _ax.set_yticklabels(_yl, fontsize=_FB); _ax.invert_yaxis()
    _ax.grid(True, axis='x', alpha=0.18, ls='--', zorder=2); _ax.tick_params(axis='x', labelsize=_FB - 1)
    _ax.spines['top'].set_visible(False); _ax.spines['right'].set_visible(False)

fig19.suptitle('Forest Plot: Effect Sizes Across Physiographic Predictors and Outcomes',
               fontsize=_FT + 1, fontweight='bold')
fig19.subplots_adjust(top=0.91, bottom=0.205, left=0.08, right=0.98)
_BT, _BH = 0.020, 0.130
fig19.add_artist(_FBox((0.01, _BT), 0.97, _BH, boxstyle='square,pad=0', facecolor='#fafafa',
                       edgecolor='#aaaaaa', linewidth=1.2, transform=fig19.transFigure, zorder=0, clip_on=False))
for _sx in (0.34, 0.67):
    fig19.add_artist(_Ln([_sx, _sx], [_BT + 0.006, _BT + _BH - 0.006], transform=fig19.transFigure,
                     color='#cccccc', lw=1.0, zorder=1, clip_on=False))
_lkw = dict(fontsize=_FB - 0.5, frameon=False, handlelength=1.2, handletextpad=0.5, borderpad=0.0,
            columnspacing=0.9, bbox_transform=fig19.transFigure)
_ay = _BT + 0.018
_zh = [_Patch(facecolor=_ZC[k], alpha=_ZA + 0.2, edgecolor='#888', lw=0.6, label=v)
       for k, v in [('negligible', 'Negligible'), ('small', 'Small'), ('medium', 'Medium'), ('large', 'Large')]]
_lg1 = fig19.legend(handles=_zh, title='Effect Size Zone', title_fontsize=_FB, loc='lower left',
                    bbox_to_anchor=(0.03, _ay), ncol=2, **_lkw); _lg1.get_title().set_fontweight('bold')
_nt = '#666666'
_ph = [_Ln([0], [0], marker='o', ls='none', color=_CFILL, markeredgecolor=_nt, markeredgewidth=1.0, markersize=9, label='Cluster25'),
       _Ln([0], [0], marker='D', ls='none', color=_EFILL['climate_class'], markeredgecolor=_nt, markeredgewidth=1.0, markersize=7, label='Climate Zone'),
       _Ln([0], [0], marker='s', ls='none', color=_EFILL['geology_class'], markeredgecolor=_nt, markeredgewidth=1.0, markersize=7, label='Geology'),
       _Ln([0], [0], marker='^', ls='none', color=_EFILL['elevation_class'], markeredgecolor=_nt, markeredgewidth=1.0, markersize=8, label='Elevation Belt')]
_lg2 = fig19.legend(handles=_ph, title='Predictor', title_fontsize=_FB, loc='lower left',
                    bbox_to_anchor=(0.36, _ay), ncol=2, **_lkw); _lg2.get_title().set_fontweight('bold')
_sh = [_Ln([0], [0], marker='o', ls='none', color='#AAAAAA', markeredgecolor=_ECS, markeredgewidth=_LWS, markersize=10, label='Significant'),
       _Ln([0], [0], marker='o', ls='none', color='#AAAAAA', markeredgecolor=_ECN, markeredgewidth=_LWN, markersize=10, label='Not significant')]
_lg3 = fig19.legend(handles=_sh, title='Significance (p < 0.05)', title_fontsize=_FB, loc='lower left',
                    bbox_to_anchor=(0.69, _ay), ncol=1, **_lkw); _lg3.get_title().set_fontweight('bold')
_p19 = savefig(fig19, MAIN_FIG / "Figure_19_forest_plot")
plt.show(); print("Figure 19 ->", ", ".join(p.name for p in _p19))

# §3 · Discussion

*Manuscript §4.* The Discussion synthesises the results; its single dedicated artwork is **Figure 20**,
a composite population-exposure panel, re-rendered from cached NB05 outputs. The large-format maps
and every individual half-donut are provided in the supplementary set (**Supp Fig 12**).

### Figure 20 · Population exposure to surface warming (composite panel)

Compact A4 panel, **fully replotted** so the maps and half-donuts read as one figure:
**(a)** national hot-spot and **(b)** cold-spot exposure maps (top) are **drawn fresh** — the noisy
Cluster25 landscape layer is dropped, the day/night city wedges use the **same orange (hot) / blue
(cold)** palette and darker-Day/lighter-Night shading as the half-donuts, scale bars are recomputed
accurately, and the most-exposed cities are annotated with ☀ day / ☾ night population. Below, a
half-donut **quincunx** — **National** (centre) framed by **Cluster 25** (top-left), **Cluster 4**
(top-right), **Cluster 13** (bottom-right) and **Cluster 9** (bottom-left), each titled with its city
count and carrying population values on segments large enough to hold them (tiny segments left
clean). One **unified legend** serves both maps and half-donuts. Full-size maps and every city /
cluster / national half-donut are in Supp Fig 12.

In [ ]:
# ── Figure 20 — FULLY REPLOTTED panel (maps drawn fresh, synced to half-donut colours) ──
import matplotlib.colors as _mc20
import matplotlib.patheffects as _pe20
from matplotlib.patches import Wedge as _W20, Patch as _P20
from matplotlib.lines import Line2D as _L20
_HOT20, _COLD20 = "#E07B00", "#2B6CB0"
_DAYF, _NIGHTF = 0.18, 0.48          # day/night shades shared by maps + half-donut Ring2
_halo20 = [_pe20.withStroke(linewidth=1.6, foreground="white")]
def _fmt20(v): return f"{v/1000:.1f}k" if v >= 1000 else f"{v:.0f}"
def _blend20(h, f):
    r, g, b = _mc20.to_rgb(h); return (r + (1 - r) * f, g + (1 - g) * f, b + (1 - b) * f)

_nigeria20 = gpd.read_file(ROOT / "data" / "inputs" / "geospatial_features" / "boundary_Nigeria.geojson")
_hd20 = pd.read_csv(RESULTS_05 / "halfdonut_city_summary.csv")
_ce20 = pd.read_csv(RESULTS_04 / "city_heat_event_summary.csv")
_NCLU = _ce20.groupby("cluster25")["city"].count().to_dict()
_hd20["cluster"] = _hd20["city"].map(dict(zip(_ce20["city"], _ce20["cluster25"])))
_coords20 = {}
for _d in sorted(p for p in RESULTS_AOI.iterdir() if p.is_dir()):
    _uc = _d / f"{_d.name}_urban_core_4326_v5.geojson"
    if _uc.exists():
        _c = gpd.read_file(_uc).geometry.union_all().centroid
        _coords20[_d.name.replace("_", " ")] = (_c.x, _c.y)

def _draw_map20(ax, spot_type, base, tag):
    _nigeria20.plot(ax=ax, facecolor="#f2f1ee", edgecolor="#333", lw=0.9, zorder=1)
    _nigeria20.boundary.plot(ax=ax, edgecolor="#222", lw=1.1, zorder=6)
    _dc, _nc = _blend20(base, _DAYF), _blend20(base, _NIGHTF)
    _sub = _hd20[_hd20.spot_type == spot_type]; _rows = {}
    for _city, (_lon, _lat) in _coords20.items():
        _s = _sub[_sub.city == _city]
        _da = float(_s[_s.period == "day"]["area_km2_50"].sum()); _na = float(_s[_s.period == "night"]["area_km2_50"].sum())
        _dp = float(_s[_s.period == "day"]["pop_50"].sum()); _np = float(_s[_s.period == "night"]["pop_50"].sum())
        _rows[_city] = dict(lon=_lon, lat=_lat, da=_da, na=_na, dp=_dp, npp=_np, ta=_da + _na, tp=_dp + _np)
    _maxp = max((r["tp"] for r in _rows.values() if r["tp"] > 0), default=1); _SC = 0.62
    for _city, r in _rows.items():
        if r["ta"] <= 0 or r["tp"] <= 0:
            ax.plot(r["lon"], r["lat"], "o", color="#c9c9c9", ms=2.5, zorder=5, alpha=0.7); continue
        r["rad"] = _SC * (max(r["tp"], 1) / _maxp) ** 0.5; _df = r["da"] / r["ta"]
        if r["da"] > 0: ax.add_patch(_W20((r["lon"], r["lat"]), r["rad"], 90, 90 + _df * 360, facecolor=_dc, edgecolor="white", lw=0.6, zorder=6))
        if r["na"] > 0: ax.add_patch(_W20((r["lon"], r["lat"]), r["rad"], 90 + _df * 360, 450, facecolor=_nc, edgecolor="white", lw=0.6, zorder=6))
        ax.plot(r["lon"], r["lat"], "ko", ms=1.6, zorder=7)
    _placed = []
    for _city, r in sorted(_rows.items(), key=lambda kv: -kv[1]["tp"]):
        if r["tp"] < 500 or "rad" not in r or len(_placed) >= 13: continue
        _px, _py = r["lon"], r["lat"] - r["rad"] - 0.05
        if any((_px - qx) ** 2 + (_py - qy) ** 2 < 0.85 ** 2 for qx, qy in _placed): continue
        ax.text(_px, _py, f"{_city}\n☀{_fmt20(r['dp'])} ☾{_fmt20(r['npp'])}", ha="center", va="top",
                fontsize=4.6, color="#1a1a1a", zorder=8, linespacing=0.95, path_effects=_halo20)
        _placed.append((_px, _py))
    _b = _nigeria20.total_bounds
    ax.set_xlim(_b[0] - 0.4, _b[2] + 0.4); ax.set_ylim(_b[1] - 0.8, _b[3] + 0.6); ax.set_aspect("equal"); ax.axis("off")
    _ml = (_b[1] + _b[3]) / 2; _dx = 200 / (111.32 * np.cos(np.radians(_ml))); _x0 = _b[0] + 0.2; _y0 = _b[1] - 0.55
    ax.plot([_x0, _x0 + _dx], [_y0, _y0], color="#111", lw=2.2, zorder=9, solid_capstyle="butt")
    ax.text(_x0 + _dx / 2, _y0 + 0.12, "200 km", ha="center", va="bottom", fontsize=5.5, zorder=9)
    ax.annotate("", xy=(_b[2] + 0.05, _b[3] + 0.35), xytext=(_b[2] + 0.05, _b[3] - 0.25), arrowprops=dict(arrowstyle="->", lw=1.1, color="#111"), zorder=9)
    ax.text(_b[2] + 0.05, _b[3] + 0.42, "N", ha="center", va="bottom", fontsize=6.5, fontweight="bold", zorder=9)
    ax.set_title(tag, fontsize=9.5, fontweight="bold", pad=2)

def _compact_halfdonut(ax, df, title=None, fs=7.0):
    r_i, r_w = [0.32, 0.60, 0.92], [0.25, 0.28, 0.28]
    def pop(st, per, col="pop_50"):
        s = df[(df.spot_type == st) & (df.period == per)]; return float(s[col].sum()) if len(s) else 0.0
    th = pop("hot", "day") + pop("hot", "night"); tc = pop("cold", "day") + pop("cold", "night"); tot = (th + tc) or 1
    hs, cs = 180 * th / tot, 180 * tc / tot
    def put(ang, r, t, size):
        a = np.radians(ang); ax.text(r * np.cos(a), r * np.sin(a), t, ha="center", va="center", fontsize=size, color="#1a1a1a", fontweight="bold", zorder=20, path_effects=_halo20)
    for st, ba, span, base in (("hot", 180 - hs, hs, _HOT20), ("cold", 0, cs, _COLD20)):
        stt = th if st == "hot" else tc
        if span < 0.5: continue
        ax.add_patch(_W20((0, 0), r_i[0] + r_w[0], ba, ba + span, width=r_w[0], facecolor=_blend20(base, 0.0), edgecolor="white", lw=1.2))
        if span >= 34: put(ba + span / 2, r_i[0] + r_w[0] / 2, _fmt20(stt), fs + 0.5)
        ang2, s2 = ba, stt or 1
        for per, fade in (("day", _DAYF), ("night", _NIGHTF)):
            pp = pop(st, per); sw2 = span * (pp / s2)
            if pp <= 0 or sw2 <= 0.5:
                ang2 += sw2; continue
            ax.add_patch(_W20((0, 0), r_i[1] + r_w[1], ang2, ang2 + sw2, width=r_w[1], facecolor=_blend20(base, fade), edgecolor="white", lw=1.0))
            if sw2 >= 24: put(ang2 + sw2 / 2, r_i[1] + r_w[1] / 2, _fmt20(pp), fs - 0.5)
            p90 = min(pop(st, per, "pop_90"), pp); p75 = min(pop(st, per, "pop_75"), pp); ang3 = ang2
            for seg, f3 in ((p90, 0.62), (max(0, p75 - p90), 0.74), (max(0, pp - p75), 0.85)):
                sw3 = sw2 * (seg / pp) if pp > 0 else 0
                if seg > 0 and sw3 > 0.3:
                    ax.add_patch(_W20((0, 0), r_i[2] + r_w[2], ang3, ang3 + sw3, width=r_w[2], facecolor=_blend20(base, f3), edgecolor="white", lw=0.6))
                    if sw3 >= 30: put(ang3 + sw3 / 2, r_i[2] + r_w[2] / 2, _fmt20(seg), fs - 1.3)
                    ang3 += sw3
            ang2 += sw2
    ax.text(0, 0.13, _fmt20(tot), ha="center", va="center", fontsize=fs + 3, fontweight="bold", color="#333")
    ax.text(0, -0.03, "pop", ha="center", va="center", fontsize=fs - 1.5, color="#777")
    if title: ax.text(0, 1.58, title, ha="center", va="center", fontsize=fs + 2, fontweight="bold")
    ax.set_xlim(-1.32, 1.32); ax.set_ylim(-0.22, 1.82); ax.axis("off"); ax.set_aspect("equal")

def _agg20(sub): return sub.groupby(["spot_type", "period"])[["pop_50", "pop_75", "pop_90"]].sum().reset_index()
def _cltitle20(cid): return f"Cluster {cid} ({_NCLU.get(cid, 0)} cities)"

fig20 = plt.figure(figsize=(8.27, 11.0))
_draw_map20(fig20.add_axes([0.015, 0.70, 0.475, 0.29]), "hot", _HOT20, "(a) Hot-spot population exposure")
_draw_map20(fig20.add_axes([0.51, 0.70, 0.475, 0.29]), "cold", _COLD20, "(b) Cold-spot population exposure")
_POS20 = {"C25": [0.02, 0.50, 0.36, 0.155], "C4": [0.62, 0.50, 0.36, 0.155], "NAT": [0.32, 0.34, 0.36, 0.16],
          "C9": [0.02, 0.175, 0.36, 0.155], "C13": [0.62, 0.175, 0.36, 0.155]}
_CID20 = {"C25": 25, "C4": 4, "C13": 13, "C9": 9}
for _k, _rect in _POS20.items():
    _ax = fig20.add_axes(_rect)
    if _k == "NAT":
        _compact_halfdonut(_ax, _agg20(_hd20), title="(c) National", fs=7.6)
    else:
        _compact_halfdonut(_ax, _agg20(_hd20[_hd20.cluster == _CID20[_k]]), title=_cltitle20(_CID20[_k]), fs=6.3)
_axl = fig20.add_axes([0.0, 0.0, 1.0, 0.14]); _axl.axis("off")
_h1 = [_P20(fc=_blend20(_HOT20, _DAYF), label="Hot · Day"), _P20(fc=_blend20(_HOT20, _NIGHTF), label="Hot · Night"),
       _P20(fc=_blend20(_COLD20, _DAYF), label="Cold · Day"), _P20(fc=_blend20(_COLD20, _NIGHTF), label="Cold · Night")]
_lg1 = _axl.legend(handles=_h1, loc="center left", bbox_to_anchor=(0.02, 0.62), ncol=2, fontsize=7.5,
                   title="Spot type × period  (darker = Day · lighter = Night)", title_fontsize=8, framealpha=0.9)
_lg1.get_title().set_fontweight("bold"); _axl.add_artist(_lg1)
_h2 = [_P20(fc=_blend20("#888", 0.62), label="L90 dense core"), _P20(fc=_blend20("#888", 0.74), label="L75 inner zone"),
       _P20(fc=_blend20("#888", 0.85), label="L50 outer extent"),
       _L20([0], [0], marker="o", mfc="none", mec="#555", ls="none", ms=10, label="circle ∝ pop (maps)")]
_lg2 = _axl.legend(handles=_h2, loc="center right", bbox_to_anchor=(0.98, 0.62), ncol=2, fontsize=7.5,
                   title="Half-donut contour rings  ·  map symbols", title_fontsize=8, framealpha=0.9)
_lg2.get_title().set_fontweight("bold")
_axl.text(0.5, 0.06, "Maps: city wedge split = Day/Night spot area, circle size ∝ population exposed  ·  "
          "Half-donuts: rings = Type → Day/Night → Contour; ☀ = day  ☾ = night",
          ha="center", va="bottom", fontsize=6.8, color="#444", transform=_axl.transAxes)
_p20 = savefig(fig20, MAIN_FIG / "Figure_20_population_exposure_panel")
plt.show(); print("Figure 20 ->", ", ".join(p.name for p in _p20))

### Supplementary Figure 12 · Large-format exposure maps & all half-donuts

The Figure 20 panel is intentionally compact; the full-resolution artwork is collected here.

- **Supp Fig 12.1.1 / 12.1.2** — national hot- / cold-spot population-exposure maps (full size).
- **Supp Fig 12.2.1–12.2.16** — per-cluster population half-donuts (by cluster id).
- **Supp Fig 12.3.1–12.3.55** — per-city population half-donuts (alphabetical).
- **Supp Fig 12.4** — national population half-donut (full size).

In [ ]:
_S12 = SUP_FIG / "12_population_exposure"; _S12.mkdir(exist_ok=True)
def _copy12(stem_src, base_dst):
    got = []
    for ext in ("svg", "pdf"):
        s = _F05 / f"{stem_src}.{ext}"
        if s.exists(): shutil.copyfile(s, _S12 / f"{base_dst}.{ext}"); got.append(ext)
    return got

_copy12("national_hot_summary_map",  "Supp_Fig_12.1.1_national_hotspot_exposure_map")
_copy12("national_cold_summary_map", "Supp_Fig_12.1.2_national_coldspot_exposure_map")
_copy12("national_population_halfdonut", "Supp_Fig_12.4_national_halfdonut")
_clids12 = sorted(int(p.stem.replace("cluster", "").split("_")[0]) for p in _F05.glob("cluster*_population_halfdonut.svg"))
for _i, _cl in enumerate(_clids12, start=1):
    _copy12(f"cluster{_cl}_population_halfdonut", f"Supp_Fig_12.2.{_i}_Cluster{_cl}_halfdonut")
_cities12 = sorted(p.stem.replace("_population_halfdonut", "") for p in _F05.glob("*_population_halfdonut.svg")
                   if not p.stem.startswith(("cluster", "national")))
for _i, _city in enumerate(_cities12, start=1):
    _copy12(f"{_city}_population_halfdonut", f"Supp_Fig_12.3.{_i}_{_city.replace(' ', '_')}_halfdonut")
print(f"Supp Fig 12: 2 maps + national half-donut + {len(_clids12)} clusters + {len(_cities12)} cities collected")